
# Working with XES Event Logs in Python

**This notebook is part of the Process Intelligence in Action (2AMI30) course**

This notebook is a hands-on introduction to the **XES (.xes) event log format**, which is widely used in **process mining**.  
You are expected to be comfortable with Python, pandas, and Jupyter notebooks.

This notebook will cover:
- Understanding the structure of an XES file (log → trace → event)
- Loading and inspecting XES files in Python
- Converting XES logs to pandas DataFrames
- Performing basic exploratory analysis on event logs

We will use the `pm4py` library, the de‑facto standard for process mining in Python.



## 1. Understanding the structure of an XES file

The **eXtensible Event Stream (XES)** format is an XML-based standard for event logs.

Conceptually, an XES log has three nested levels:

1. **Log** – the whole dataset
2. **Trace** – one process instance
3. **Event** – one step in the process

Each level can have **attributes**:
- Events typically have: `concept:name`, `time:timestamp`, `org:resource`
- Traces often have: `case_id`, `customer`, `variant`

Think of a trace as a sequence of time-ordered events describing *what happened* for a single case.



## 2. Required libraries

We use **pm4py** for reading and analyzing XES logs.

If pm4py is not installed, run the following (once):
```bash
pip install pm4py
```


In [ ]:

import pm4py
import pandas as pd

from pm4py.objects.log.importer.xes import importer as xes_importer
from pm4py.objects.conversion.log import converter as log_converter



## 3. Loading an XES file

Place your `.xes` file in the same directory as this notebook, or update the path below.


In [ ]:

# Update this path to your XES file
xes_path = "Supermarket_data\Supermarket_Customer.xes"

log = xes_importer.apply(xes_path)

print(type(log))
print(f"Number of traces: {len(log)}")



## 4. Inspecting traces and events

An imported XES log behaves like a **list of traces**.
Each trace behaves like a **list of events**.


In [ ]:

# Inspect the first trace
first_trace = log[0]

print(type(first_trace))
print(f"Number of events in first trace: {len(first_trace)}")
print("Trace attributes:")
print(first_trace.attributes)


In [ ]:
log[0]

In [ ]:

# Inspect the first event of the first trace
first_event = first_trace[0]

print(type(first_event))
print(first_event)



## 5. Common event attributes

Typical event attributes you should always look for:
- `concept:name` – the activity name
- `time:timestamp` – when the event occurred
- `org:resource` – who executed it (if available)

Not all logs contain all attributes.


In [ ]:

# Collect all event attribute keys in the log
attribute_keys = set()

for trace in log:
    for event in trace:
        attribute_keys.update(event.keys())

attribute_keys



## 6. Converting an XES log to a pandas DataFrame

For data science workflows, it is often convenient to work with a **tabular representation**.

pm4py provides a built-in conversion.


In [ ]:

log_df = log_converter.apply(log, variant=log_converter.Variants.TO_DATA_FRAME)

log_df.head()



### Interpreting the DataFrame

Typical columns:
- `case:concept:name` → case ID
- `concept:name` → activity
- `time:timestamp` → event time

Each row corresponds to **one event**.
Multiple rows with the same case ID form a trace.



## 7. Basic exploratory analysis


In [ ]:

# Number of events
len(log_df)


In [ ]:

# Number of unique cases
log_df['case:concept:name'].nunique()


In [ ]:

# Most frequent activities
log_df['concept:name'].value_counts().head(10)


In [ ]:

# Trace length distribution
trace_lengths = log_df.groupby('case:concept:name').size()
trace_lengths.describe()



## 8. Event ordering and timestamps

Event order **must always be derived from timestamps**, not row order.


In [ ]:

# Ensure correct ordering
log_df_sorted = log_df.sort_values(
    ['case:concept:name', 'time:timestamp']
)

log_df_sorted.head()


## 9. Computing case duration

Case duration is the time between the first and last event of a case (trace).
This is one of the most basic but informative performance indicators.

In [ ]:
# Compute case start and end times
case_times = (
    log_df_sorted
    .groupby('case:concept:name')['time:timestamp']
    .agg(['min', 'max'])
)

# Compute duration in seconds
case_times['duration_seconds'] = (
    case_times['max'] - case_times['min']
).dt.total_seconds()

case_times.head()

In [ ]:
# Plot case duration histogram
import matplotlib.pyplot as plt

plt.hist(case_times['duration_seconds'], bins=50)
plt.xlabel("Case duration (seconds)")
plt.ylabel("Number of cases")
plt.title("Distribution of Case Durations")
plt.show()

## 10. Variants: identifying execution paths

A variant is a unique sequence of activities executed by a case.
Variants are essential for understanding process complexity and deviations.

In [ ]:
# Build activity sequences per case
variants = (
    log_df_sorted
    .groupby('case:concept:name')['concept:name']
    .apply(tuple)
)

variants.head()

In [ ]:
# Count variant frequencies
variant_counts = variants.value_counts().reset_index()
variant_counts.columns = ['variant', 'frequency']

variant_counts.head(10)


## 11. Activity frequency analysis

Activity frequencies provide a first structural view of the process.

In [ ]:
# Bar chart of activity frequencies
activity_counts = log_df['concept:name'].value_counts()

activity_counts.plot(kind='bar')
plt.xlabel("Activity")
plt.ylabel("Frequency")
plt.title("Activity Frequencies")
plt.show()


## 12. Temporal analysis: arrivals over time

Event logs often exhibit strong temporal patterns:

- business hours
- daily/weekly seasonality
- batching effects

In [ ]:
# Hourly arrivals
log_df['hour'] = log_df['time:timestamp'].dt.hour

hourly_arrivals = log_df.groupby('hour').size()

hourly_arrivals.plot(kind='bar')
plt.xlabel("Hour of day")
plt.ylabel("Number of events")
plt.title("Hourly Event Arrivals")
plt.show()

In [ ]:
# Daily arrivals
log_df['date'] = log_df['time:timestamp'].dt.date

daily_arrivals = log_df.groupby('date').size()

daily_arrivals.plot()
plt.xlabel("Date")
plt.ylabel("Number of events")
plt.title("Daily Event Arrivals")
plt.show()


## 13. Dotted chart (process timeline visualization)

A dotted chart visualizes:

- x-axis → time
- y-axis → cases
- dots → events, colored by activity

This reveals concurrency, batching, and bottlenecks.

In [ ]:
import seaborn as sns

In [ ]:

start = pd.Timestamp("2026-01-26 14:00:00", tz="UTC")
end   = pd.Timestamp("2026-01-26 15:00:00", tz="UTC")

log_df_day = log_df[
    (log_df['time:timestamp'] >= start) &
    (log_df['time:timestamp'] < end)
]



In [ ]:
dotted_chart = sns.scatterplot(log_df_day.sort_values(by="time:timestamp"), x='time:timestamp', y='case:concept:name', hue='concept:name', alpha=.7)
sns.move_legend(dotted_chart, "upper left", bbox_to_anchor=(1, 1))
plt.xticks(rotation=30, ha="right")
plt.grid()
dotted_chart;

## 14. Process discovery with pm4py

**Process discovery** automatically derives a process model from event data.
We start with the **Directly-Follows Graph (DFG)** and discover a **Petri net**.

In [ ]:
# Discover and visualize DFG
dfg, start_activities, end_activities = pm4py.discover_dfg(log)

pm4py.view_dfg(dfg, start_activities, end_activities)


In [ ]:
# Discover a Petri net (Inductive Miner)
net, initial_marking, final_marking = pm4py.discover_petri_net_inductive(log)

pm4py.view_petri_net(net, initial_marking, final_marking)


## Other things to consider when working with XES logs

- Missing or inconsistent timestamps
- Multiple events with identical timestamps
- Multiple lifecycle events (start / complete)
- Very large logs that do not fit in memory
- Assuming every case follows the same path

Always *inspect before modeling*.


## How do queues affect the abandonment of carts?

In [ ]:
# the log has an explicit 'Abandon cart and leave' activity which makes this easier
abandoned_ids = set(log_df[log_df['concept:name'] == 'Abandon cart and leave']['case:concept:name'])
completed_ids = set(log_df[log_df['concept:name'] == 'Complete Payment']['case:concept:name'])

print(f"abandoned: {len(abandoned_ids)}")
print(f"completed: {len(completed_ids)}")
print(f"abandonment rate: {len(abandoned_ids) / (len(abandoned_ids) + len(completed_ids)) * 100:.1f}%")


In [ ]:
# abandonment rate per 15 min
log_df['t15'] = log_df['time:timestamp'].dt.floor('15min')

hourly_abandon = log_df[log_df['concept:name'] == 'Abandon cart and leave'].groupby('t15').size()
hourly_checkout = log_df[log_df['concept:name'].isin(['Enter Queue', 'Abandon cart and leave'])].groupby('t15').size()
abandon_rate = (hourly_abandon / hourly_checkout).rename('abandonment_rate')

avg_queue = (
    log_df[log_df['concept:name'] == 'Enter Queue']
    .groupby('t15')['mC']
    .mean()
    .rename('avg_queue_depth')
)

every15 = pd.concat([abandon_rate, avg_queue], axis=1).dropna()
print(every15)


In [ ]:
# pick a week — change week_start to look at a different one
week_start = pd.Timestamp("2026-02-08", tz="UTC")
week_end = week_start + pd.Timedelta(days=7)

week_df = log_df[
    (log_df['time:timestamp'] >= week_start) &
    (log_df['time:timestamp'] < week_end)
].copy()

# time of day as a float
week_df['tod'] = week_df['time:timestamp'].dt.hour + week_df['time:timestamp'].dt.minute / 60
week_df['t15'] = (week_df['tod'] * 4).round() / 4
week_df['day'] = week_df['time:timestamp'].dt.date

days = sorted(week_df['day'].unique())

fig, axes = plt.subplots(1, 7, figsize=(21, 4), sharey=False)

for ax, day in zip(axes, days):
    day_df = week_df[week_df['day'] == day]

    abandon = day_df[day_df['concept:name'] == 'Abandon cart and leave'].groupby('t15').size()
    checkout = day_df[day_df['concept:name'].isin(['Enter Queue', 'Abandon cart and leave'])].groupby('t15').size()
    rate = (abandon / checkout).fillna(0)

    queue = day_df[day_df['concept:name'] == 'Enter Queue'].groupby('t15')['mC'].mean()

    ax2 = ax.twinx()
    ax.bar(rate.index, rate.values, width=0.2, color='tomato', alpha=0.6, label='abandon rate')
    ax2.plot(queue.index, queue.values, color='steelblue', linewidth=1.5, label='avg queue depth')

    ax.set_title(str(day), fontsize=8)
    ax.set_xlabel('Hour', fontsize=7)
    ax.tick_params(axis='x', labelsize=7)
    ax.tick_params(axis='y', labelsize=7, colors='tomato')
    ax2.tick_params(axis='y', labelsize=7, colors='steelblue')

axes[0].set_ylabel('Abandonment rate', color='tomato', fontsize=8)
axes[-1].set_ylabel('Avg queue depth', color='steelblue', fontsize=8)

plt.suptitle('Abandonment rate vs queue depth per day (week of 2026-01-26)', fontsize=10)
plt.tight_layout()
plt.show()


In [ ]:
abandonment_by_hour = (
    log_df[log_df['concept:name'] == 'Abandon cart and leave']
    .groupby('hour')
    .size()
    .rename('n_abandonments')
)

print(abandonment_by_hour)


## How customer behaviour impacts queue selection dynamics? 

In [ ]:
## How does customer behavior impact queue selection dynamics?

import re

def parse_col(col_str):
    if pd.isna(col_str):
        return []
    matches = re.findall(r'\((\d+),(\d+),(\d+),(true|false)\)', str(col_str))
    return [(int(m[0]), int(m[1]), int(m[2]), m[3] == 'true') for m in matches]

# verify parsing on a real example
eq_events = log_df[log_df['concept:name'] == 'Enter Queue'].copy()
print("Example col string:")
print(eq_events['col'].dropna().iloc[0])
print()
print("Parsed:")
print(parse_col(eq_events['col'].dropna().iloc[0]))


In [ ]:
# build the selection table — only keep rows where the customer had a real choice
results = []

for _, row in eq_events.dropna(subset=['col', 'cid', 'mC']).iterrows():
    counters = parse_col(row['col'])
    open_counters = [c for c in counters if c[3]]

    if len(open_counters) < 2:
        continue

    chosen_cid = int(row['cid'])
    chosen = next((c for c in counters if c[0] == chosen_cid), None)
    if chosen is None:
        continue

    min_queue_len = min(c[1] for c in open_counters)
    min_items     = min(c[2] for c in open_counters)

    results.append({
        'n_open':          len(open_counters),
        'chosen_queue_len': chosen[1],
        'chosen_items':     chosen[2],
        'min_queue_len':    min_queue_len,
        'min_items':        min_items,
        'queue_gap':        chosen[1] - min_queue_len,
        'items_gap':        chosen[2] - min_items,
        'own_basket':       row['items'],
    })

selection_df = pd.DataFrame(results)
print(f"Queue entries where customer had a real choice (>1 counter open): {len(selection_df)}")
print(f"Chose the shortest queue: {(selection_df['queue_gap'] == 0).mean()*100:.1f}%")
print()
print(selection_df[['queue_gap', 'items_gap']].describe().round(1))


Does queue length influence which queue customers pick?

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

gap_counts = selection_df['queue_gap'].value_counts().sort_index()
bars = ax.bar(gap_counts.index.astype(str), gap_counts.values, color='steelblue', width=0.5)

for bar, val in zip(bars, gap_counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 30,
            f'{val:,}', ha='center', va='bottom', fontsize=9)

pct_optimal = (selection_df['queue_gap'] == 0).mean() * 100
ax.set_xlabel('Extra customers in chosen queue vs shortest available queue (gap)', fontsize=11)
ax.set_ylabel('Number of customers', fontsize=11)
ax.set_title(f'Queue length gap — {pct_optimal:.1f}% of customers chose the shortest available queue\n'
             f'(only cases where ≥2 counters were open, n={len(selection_df):,})', fontsize=11)
plt.tight_layout()
plt.show()


Does the number of items in a queue affect which queue customers pick?

In [ ]:
# when queue lengths are tied, items in each queue differentiate
equal_len = selection_df[selection_df['queue_gap'] == 0].copy()
pct_min_items = (equal_len['items_gap'] == 0).mean() * 100

print(f"Cases where all open queues had equal headcount: {len(equal_len)}")
print(f"Of those, chose the counter with fewest items in queue: {pct_min_items:.1f}%")

equal_len['own_basket_group'] = pd.cut(
    equal_len['own_basket'],
    bins=[0, 10, 30, 200],
    labels=['small\n(<10 items)', 'medium\n(10–30)', 'large\n(>30)']
)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# left: items gap distribution
equal_len['items_gap'].clip(upper=80).plot(
    kind='hist', bins=25, ax=axes[0], color='steelblue', edgecolor='white'
)
axes[0].set_xlabel('Extra items in chosen queue vs lightest available queue', fontsize=10)
axes[0].set_ylabel('Number of customers', fontsize=10)
axes[0].set_title(f'Items in queue gap (equal headcount cases)\n'
                  f'{pct_min_items:.1f}% picked the counter with fewest items', fontsize=10)

# right: basket size vs items gap
pct_by_basket = (
    equal_len.groupby('own_basket_group', observed=True)
    .apply(lambda x: (x['items_gap'] == 0).mean() * 100)
    .rename('pct_chose_min_items')
)

pct_by_basket.plot(kind='bar', ax=axes[1], color='steelblue', width=0.5)
axes[1].set_xlabel("Customer's own basket size", fontsize=10)
axes[1].set_ylabel('% who chose the counter with fewest items', fontsize=10)
axes[1].set_title("Do customers with smaller baskets\npick queues with fewer items more carefully?", fontsize=10)
axes[1].tick_params(axis='x', rotation=0)
axes[1].set_ylim(0, 100)

for bar, val in zip(axes[1].patches, pct_by_basket.values):
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1.5,
                 f'{val:.1f}%', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()


left:
Only looks at cases where multiple queues had the same number of people
The spike at 0 means when headcount was tied, 90.4% still picked the queue with fewest items
Takeaway: when customers can't distinguish by headcount, they do seem to pick the lighter queue

right:
All three bars are ~90% regardless of whether the customer had a small or large basket
Takeaway: the customer's own basket size has no effect on how they pick a queue

In [ ]:
# chosen vs available per queue entry

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

max_len = int(selection_df[['chosen_queue_len', 'min_queue_len']].max().max()) + 1

# left: queue length, chosen vs shortest
axes[0].hist(selection_df['min_queue_len'], bins=range(0, max_len + 1),
             alpha=0.6, label='Shortest available', color='steelblue')
axes[0].hist(selection_df['chosen_queue_len'], bins=range(0, max_len + 1),
             alpha=0.6, label='Actually chosen', color='tomato')
axes[0].set_xlabel('Queue length (number of people)', fontsize=11)
axes[0].set_ylabel('Number of customers', fontsize=11)
axes[0].set_title('Queue length: chosen vs shortest available', fontsize=11)
axes[0].legend(fontsize=10)

# right: items, chosen vs fewest
axes[1].hist(selection_df['min_items'].clip(upper=150), bins=25,
             alpha=0.6, label='Fewest items available', color='steelblue')
axes[1].hist(selection_df['chosen_items'].clip(upper=150), bins=25,
             alpha=0.6, label='Actually chosen', color='tomato')
axes[1].set_xlabel('Items in queue (total remaining to scan)', fontsize=11)
axes[1].set_ylabel('Number of customers', fontsize=11)
axes[1].set_title('Items in queue: chosen vs minimum available', fontsize=11)
axes[1].legend(fontsize=10)

plt.suptitle('Do customers choose queues based on headcount or items in queue?\n'
             f'(n={len(selection_df):,} entries where ≥2 counters were open)', fontsize=12)
plt.tight_layout()
plt.show()


left: <br>
Blue = how long the shortest available queue was at that moment <br>
Red = what the customer actually joined <br>
Blue has a bigger spike at 0 than red → there were often empty queues available, but customers didn't always take them
Red has bigger bars at 1, 2, 3 than blue → customers frequently joined longer queues when shorter ones existed
Takeaway: customers don't always pick the shortest queue

right:
Same idea but for items instead of people
Blue and red are nearly identical, both piled up near 0
Takeaway: chosen and available are almost the same — but this is probably just because short queues naturally have few items too, not because customers are reading item counts

In [ ]:
# summarise which dimension customers actually tracked
conditions = {
    'Shortest queue\n& fewest items':  (selection_df['queue_gap'] == 0) & (selection_df['items_gap'] == 0),
    'Shortest queue,\nnot fewest items': (selection_df['queue_gap'] == 0) & (selection_df['items_gap'] > 0),
    'Fewest items,\nnot shortest queue': (selection_df['queue_gap'] > 0) & (selection_df['items_gap'] == 0),
    'Neither':                          (selection_df['queue_gap'] > 0) & (selection_df['items_gap'] > 0),
}

counts = {label: cond.sum() for label, cond in conditions.items()}
colors = ['steelblue', 'cornflowerblue', 'lightsalmon', 'tomato']

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(counts.keys(), counts.values(), color=colors, width=0.5)

for bar, count in zip(bars, counts.values()):
    pct = count / len(selection_df) * 100
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 20,
            f'{count:,}\n({pct:.1f}%)', ha='center', va='bottom', fontsize=9)

ax.set_ylabel('Number of customers', fontsize=11)
ax.set_title('What did customers optimise for when choosing a queue?', fontsize=12)
ax.tick_params(axis='x', labelsize=9)
plt.tight_layout()
plt.show()


Step 1 — Filter and prep: Keep only Enter Queue events (the moments a customer makes a choice), extract which counters were open from col, and drop events where only 1 counter was open (no real choice).

Step 2 — For each (hour, counter) pair: exposed is the subset of events in that hour where counter k was open. Each of those events is an independent Bernoulli trial where the customer picks counter k with probability 1/n_open. We compute actual (how many times k was chosen), expected (sum of 1/n_open across exposed events — the expected picks under no preference), and variance (sum of p*(1-p)). The test statistic is z = (actual - expected) / sqrt(variance), the standard normal approximation for a sum of independent Bernoullis with different probabilities (Poisson-binomial distribution), with a two-sided p-value.

Step 3 — Ratio: actual / expected is the preference ratio. 1.0 means chosen exactly as often as chance predicts, above 1 means over-chosen, below 1 means avoided.

Steps 4–5 — Visualization and output: The heatmap shows the ratio for every counter × hour combination, the bar chart collapses across hours for an overall view, and the significance table lists only cells where the preference is statistically significant (p < 0.05).

In [ ]:
import numpy as np
from scipy.stats import norm

# isolate queue entry events
eq = (
    log_df[log_df['concept:name'] == 'Enter Queue']
    .dropna(subset=['col', 'cid'])
    .copy()
)
eq['hour'] = eq['time:timestamp'].dt.hour
eq['cid'] = eq['cid'].astype(int)
eq['open'] = eq['col'].apply(lambda c: [x[0] for x in parse_col(c) if x[3]])
eq['n_open'] = eq['open'].apply(len)
eq = eq[eq['n_open'] >= 2]  # only events with a real choice

all_counters = sorted({c for counters in eq['open'] for c in counters})

# per (hour, counter): actual vs expected picks
records = []
for hour, grp in eq.groupby('hour'):
    for counter in all_counters:
        exposed = grp[grp['open'].apply(lambda cs: counter in cs)]
        if len(exposed) == 0:
            continue

        p_i = 1 / exposed['n_open']
        actual   = (exposed['cid'] == counter).sum()
        expected = p_i.sum()
        variance = (p_i * (1 - p_i)).sum()

        z = (actual - expected) / np.sqrt(variance)
        pvalue = 2 * norm.sf(abs(z))

        records.append({
            'hour':     hour,
            'counter':  counter,
            'actual':   actual,
            'expected': expected,
            'ratio':    actual / expected,
            'pvalue':   pvalue,
        })

pref = pd.DataFrame(records)

# heatmap of ratio by counter and hour
pivot = pref.pivot(index='counter', columns='hour', values='ratio')

fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(
    pivot, ax=ax, center=1.0, cmap='RdYlGn',
    annot=True, fmt='.2f', linewidths=0.4,
    cbar_kws={'label': 'actual / expected (1.0 = no preference)'}
)
ax.set_title('Counter selection preference by hour')
ax.set_xlabel('Hour of day')
ax.set_ylabel('Counter ID')
plt.tight_layout()
plt.show()

# overall preference across hours
overall = (
    pref.groupby('counter')[['actual', 'expected']]
    .sum()
    .assign(ratio=lambda d: d['actual'] / d['expected'])
    .reset_index()
)

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(overall['counter'].astype(str), overall['ratio'], width=0.5, color='steelblue')
ax.axhline(1.0, color='red', linestyle='--', linewidth=1, label='no preference')
for bar, val in zip(bars, overall['ratio']):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f'{val:.2f}', ha='center', va='bottom', fontsize=9)
ax.set_xlabel('Counter ID')
ax.set_ylabel('Actual / expected picks')
ax.set_title('Overall counter preference (controlling for availability)')
ax.legend()
plt.tight_layout()
plt.show()

# print significant cells
sig = pref[pref['pvalue'] < 0.05].sort_values('pvalue')
print(f"{len(sig)} significant (hour, counter) pairs out of {len(pref)}")
print(sig.round(3).to_string(index=False))


Which counters are most probably to be opened at which hour?

In [ ]:
# P(counter k is open) at each hour

all_counters = sorted({c for cs in eq['open'] for c in cs})
hours = sorted(eq['hour'].unique())

rows = []
for hour in hours:
    grp = eq[eq['hour'] == hour]
    total = len(grp)
    for counter in all_counters:
        n_open = grp['open'].apply(lambda cs: counter in cs).sum()
        rows.append({'hour': hour, 'counter': counter, 'prob': n_open / total})

prob_df = pd.DataFrame(rows)
pivot = prob_df.pivot(index='counter', columns='hour', values='prob')

fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(pivot, ax=ax, cmap='Blues', annot=True, fmt='.2f',
            linewidths=0.4, vmin=0, vmax=1,
            cbar_kws={'label': 'P(open)'})
ax.set_title('Probability of each counter being open, by hour')
ax.set_xlabel('Hour of day')
ax.set_ylabel('Counter ID')
plt.tight_layout()
plt.show()


In [ ]:
# timestamps for the two events of interest, per case
enter_queue = (
    log_df[log_df['concept:name'] == 'Enter Queue']
    [['case:concept:name', 'time:timestamp']]
    .rename(columns={'time:timestamp': 'queue_time'})
)

scan_item = (
    log_df[log_df['concept:name'] == 'Scan Item']
    [['case:concept:name', 'time:timestamp']]
    .rename(columns={'time:timestamp': 'scan_time'})
)

# for each case, keep only the first Scan Item that occurs after Enter Queue
merged = enter_queue.merge(scan_item, on='case:concept:name')
merged = merged[merged['scan_time'] > merged['queue_time']]
first_scan = merged.groupby('case:concept:name').first().reset_index()

first_scan['wait_seconds'] = (first_scan['scan_time'] - first_scan['queue_time']).dt.total_seconds()
first_scan['hour'] = first_scan['queue_time'].dt.hour

# statistics per hour
stats = (
    first_scan.groupby('hour')['wait_seconds']
    .agg(n='count', mean='mean', median='median', std='std',
         p25=lambda x: x.quantile(0.25), p75=lambda x: x.quantile(0.75))
    .round(1)
)
print(stats)

# plot
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(stats.index, stats['mean'], color='steelblue', width=0.5, label='mean')
ax.errorbar(stats.index, stats['mean'], yerr=stats['std'], fmt='none',
            color='black', capsize=4, linewidth=1, label='±1 std')
ax.set_xlabel('Hour of day (at queue entry)')
ax.set_ylabel('Wait time (seconds)')
ax.set_title('Average waiting time from Enter Queue to first Scan Item, by hour')
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Boxplot: waiting time from Enter Queue to first Scan Item, by hour
fig, ax = plt.subplots(figsize=(10, 4))

groups = [first_scan[first_scan['hour'] == h]['wait_seconds'].values
          for h in sorted(first_scan['hour'].unique())]
labels = sorted(first_scan['hour'].unique())

ax.boxplot(groups, labels=labels, patch_artist=True,
           boxprops=dict(facecolor='steelblue', alpha=0.6),
           medianprops=dict(color='black', linewidth=2),
           flierprops=dict(marker='o', markersize=3, alpha=0.3))

ax.set_xlabel('Hour of day (at queue entry)')
ax.set_ylabel('Wait time (seconds)')
ax.set_title('Waiting time from Enter Queue to first Scan Item, by hour')
plt.tight_layout()
plt.show()


## Does cart abandonment lead to more cast abandonment?

In [ ]:
# Filter for cart abandonments between 14:00 and 15:00
abandon_14_15 = log_df[
    (log_df['concept:name'] == 'Abandon cart and leave') &
    (log_df['time:timestamp'].dt.hour == 14)
].copy()

# Extract date and time of day for plotting
abandon_14_15['date'] = abandon_14_15['time:timestamp'].dt.date
abandon_14_15['minutes_in_hour'] = abandon_14_15['time:timestamp'].dt.minute + abandon_14_15['time:timestamp'].dt.second / 60

# Map dates to numeric indices for plotting
unique_dates = sorted(abandon_14_15['date'].unique())
date_to_idx = {date: idx for idx, date in enumerate(unique_dates)}
abandon_14_15['date_idx'] = abandon_14_15['date'].map(date_to_idx)

# Create the plot with compact date display
fig, ax = plt.subplots(figsize=(14, 8))

ax.scatter(
    abandon_14_15['minutes_in_hour'],
    abandon_14_15['date_idx'],
    alpha=0.6,
    s=100,
    color='tomato',
    edgecolors='darkred',
    linewidth=1
)

# Set y-axis labels to show actual dates
ax.set_yticks(range(len(unique_dates)))
ax.set_yticklabels([str(d) for d in unique_dates], fontsize=9)

ax.set_xlabel('Minutes within the 14:00-15:00 hour', fontsize=11)
ax.set_ylabel('Date', fontsize=11)
ax.set_title('Cart Abandonments between 14:00-15:00 by Day', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')
ax.set_xticks(range(0, 61, 5))

plt.tight_layout()
plt.show()

print(f"Total cart abandonments between 14:00-15:00: {len(abandon_14_15)}")
print(f"Days with abandonments: {abandon_14_15['date'].nunique()}")

In [ ]:
# Filter for cart abandonments between 14:00 and 15:00
abandon_14_15 = log_df[
    (log_df['concept:name'] == 'Abandon cart and leave') &
    (log_df['time:timestamp'].dt.hour == 15)
].copy()

# Extract date and time of day for plotting
abandon_14_15['date'] = abandon_14_15['time:timestamp'].dt.date
abandon_14_15['minutes_in_hour'] = abandon_14_15['time:timestamp'].dt.minute + abandon_14_15['time:timestamp'].dt.second / 60

# Map dates to numeric indices for plotting
unique_dates = sorted(abandon_14_15['date'].unique())
date_to_idx = {date: idx for idx, date in enumerate(unique_dates)}
abandon_14_15['date_idx'] = abandon_14_15['date'].map(date_to_idx)

# Create the plot with compact date display
fig, ax = plt.subplots(figsize=(14, 8))

ax.scatter(
    abandon_14_15['minutes_in_hour'],
    abandon_14_15['date_idx'],
    alpha=0.6,
    s=100,
    color='tomato',
    edgecolors='darkred',
    linewidth=1
)

# Set y-axis labels to show actual dates
ax.set_yticks(range(len(unique_dates)))
ax.set_yticklabels([str(d) for d in unique_dates], fontsize=9)

ax.set_xlabel('Minutes within the 15:00-16:00 hour', fontsize=11)
ax.set_ylabel('Date', fontsize=11)
ax.set_title('Cart Abandonments between 14:00-15:00 by Day', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')
ax.set_xticks(range(0, 61, 5))

plt.tight_layout()
plt.show()

print(f"Total cart abandonments between 15:00-16:00: {len(abandon_14_15)}")
print(f"Days with abandonments: {abandon_14_15['date'].nunique()}")

In [ ]:
# Filter for cart abandonments between 14:00 and 15:00
abandon_14_15 = log_df[
    (log_df['concept:name'] == 'Abandon cart and leave') &
    (log_df['time:timestamp'].dt.hour == 19)
].copy()

# Extract date and time of day for plotting
abandon_14_15['date'] = abandon_14_15['time:timestamp'].dt.date
abandon_14_15['minutes_in_hour'] = abandon_14_15['time:timestamp'].dt.minute + abandon_14_15['time:timestamp'].dt.second / 60

# Map dates to numeric indices for plotting
unique_dates = sorted(abandon_14_15['date'].unique())
date_to_idx = {date: idx for idx, date in enumerate(unique_dates)}
abandon_14_15['date_idx'] = abandon_14_15['date'].map(date_to_idx)

# Create the plot with compact date display
fig, ax = plt.subplots(figsize=(14, 8))

ax.scatter(
    abandon_14_15['minutes_in_hour'],
    abandon_14_15['date_idx'],
    alpha=0.6,
    s=100,
    color='tomato',
    edgecolors='darkred',
    linewidth=1
)

# Set y-axis labels to show actual dates
ax.set_yticks(range(len(unique_dates)))
ax.set_yticklabels([str(d) for d in unique_dates], fontsize=9)

ax.set_xlabel('Minutes within the 19:00-20:00 hour', fontsize=11)
ax.set_ylabel('Date', fontsize=11)
ax.set_title('Cart Abandonments between 19:00-20:00 by Day', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')
ax.set_xticks(range(0, 61, 5))

plt.tight_layout()
plt.show()

print(f"Total cart abandonments between 19:00-20:00: {len(abandon_14_15)}")
print(f"Days with abandonments: {abandon_14_15['date'].nunique()}")

## Does the number of customers lead to higher cart abandonment rates?

In [ ]:
# For each day: mini plot with customers in store and cart abandonments over time
# Arranged in a 7-column grid so we can spot weekly patterns

HOUR_START, HOUR_END = 14, 15  # focus on store hours
BIN_MIN = 5                   # 15-minute bins
NCOLS = 7                      # one column per weekday

all_dates = sorted(log_df['time:timestamp'].dt.date.unique())
num_days  = len(all_dates)
nrows     = (num_days + NCOLS - 1) // NCOLS

fig, axes = plt.subplots(nrows, NCOLS, figsize=(21, 3.5 * nrows), squeeze=False)
axes_flat = axes.flatten()

tz = log_df['time:timestamp'].dt.tz

line_cust  = None
line_aband = None

all_customers = []
all_abandon   = []

for day in all_dates:
    day_data = log_df[log_df['time:timestamp'].dt.date == day]
    time_bins = pd.date_range(
        start=pd.Timestamp(str(day), tz=tz) + pd.Timedelta(hours=HOUR_START),
        end  =pd.Timestamp(str(day), tz=tz) + pd.Timedelta(hours=HOUR_END),
        freq =f'{BIN_MIN}min'
    )
    customers_per_bin    = []
    abandonments_per_bin = []
    for i in range(len(time_bins) - 1):
        bin_start, bin_end = time_bins[i], time_bins[i + 1]
        entered = set(day_data.loc[(day_data['concept:name'] == 'Enter store') & (day_data['time:timestamp'] < bin_end), 'case:concept:name'])
        exited  = set(day_data.loc[(day_data['concept:name'].isin(['Complete Payment', 'Abandon cart and leave'])) & (day_data['time:timestamp'] < bin_start), 'case:concept:name'])
        customers_per_bin.append(len(entered - exited))
        abandonments_per_bin.append(day_data[(day_data['concept:name'] == 'Abandon cart and leave') & (day_data['time:timestamp'] >= bin_start) & (day_data['time:timestamp'] < bin_end)].shape[0])
    all_customers.append(customers_per_bin)
    all_abandon.append(abandonments_per_bin)

ylim_cust   = (0, max(max(c) for c in all_customers) + 1)
ylim_abandon = (0, max(max(a) for a in all_abandon)  + 1)

for idx, (day, customers_per_bin, abandonments_per_bin) in enumerate(zip(all_dates, all_customers, all_abandon)):
    ax = axes_flat[idx]

    x = [HOUR_START + i * BIN_MIN / 60 for i in range(len(customers_per_bin))]

    line_cust, = ax.plot(x, customers_per_bin, color='steelblue', linewidth=1.5)
    ax.set_ylabel('Customers', fontsize=6, color='steelblue')
    ax.tick_params(axis='y', labelcolor='steelblue', labelsize=5)
    ax.set_xlim(HOUR_START, HOUR_END)
    ax.set_ylim(*ylim_cust)
    ax.set_xticks(range(HOUR_START, HOUR_END + 1))
    ax.set_xticklabels([str(h) for h in range(HOUR_START, HOUR_END + 1)], fontsize=5)
    ax.grid(True, alpha=0.3)
    ax.set_title(str(day), fontsize=7, fontweight='bold', pad=2)

    ax2 = ax.twinx()
    line_aband, = ax2.plot(x, abandonments_per_bin, color='tomato', linewidth=1.5, linestyle='--')
    ax2.set_ylabel('Abandonments', fontsize=6, color='tomato')
    ax2.tick_params(axis='y', labelcolor='tomato', labelsize=5)
    ax2.set_ylim(*ylim_abandon)

for idx in range(num_days, len(axes_flat)):
    axes_flat[idx].set_visible(False)

fig.legend(
    handles=[line_cust, line_aband],
    labels=['Customers in store', 'Cart abandonments'],
    loc='lower center', ncol=2, fontsize=10,
    bbox_to_anchor=(0.5, 0), bbox_transform=fig.transFigure
)

plt.suptitle('Customers in store vs cart abandonments per day (15-min bins, 14:00–15:00)',
             fontsize=12, fontweight='bold')
plt.tight_layout(rect=[0, 0.04, 1, 0.99])
plt.show()

print(f"Plotted {num_days} days")


In [ ]:
# For each day: mini plot with customers in store and cart abandonments over time
# Arranged in a 7-column grid so we can spot weekly patterns

HOUR_START, HOUR_END = 15, 16  # focus on store hours
BIN_MIN = 5                   # 15-minute bins
NCOLS = 7                      # one column per weekday

all_dates = sorted(log_df['time:timestamp'].dt.date.unique())
num_days  = len(all_dates)
nrows     = (num_days + NCOLS - 1) // NCOLS

fig, axes = plt.subplots(nrows, NCOLS, figsize=(21, 3.5 * nrows), squeeze=False)
axes_flat = axes.flatten()

tz = log_df['time:timestamp'].dt.tz

line_cust  = None
line_aband = None

all_customers = []
all_abandon   = []

for day in all_dates:
    day_data = log_df[log_df['time:timestamp'].dt.date == day]
    time_bins = pd.date_range(
        start=pd.Timestamp(str(day), tz=tz) + pd.Timedelta(hours=HOUR_START),
        end  =pd.Timestamp(str(day), tz=tz) + pd.Timedelta(hours=HOUR_END),
        freq =f'{BIN_MIN}min'
    )
    customers_per_bin    = []
    abandonments_per_bin = []
    for i in range(len(time_bins) - 1):
        bin_start, bin_end = time_bins[i], time_bins[i + 1]
        entered = set(day_data.loc[(day_data['concept:name'] == 'Enter store') & (day_data['time:timestamp'] < bin_end), 'case:concept:name'])
        exited  = set(day_data.loc[(day_data['concept:name'].isin(['Complete Payment', 'Abandon cart and leave'])) & (day_data['time:timestamp'] < bin_start), 'case:concept:name'])
        customers_per_bin.append(len(entered - exited))
        abandonments_per_bin.append(day_data[(day_data['concept:name'] == 'Abandon cart and leave') & (day_data['time:timestamp'] >= bin_start) & (day_data['time:timestamp'] < bin_end)].shape[0])
    all_customers.append(customers_per_bin)
    all_abandon.append(abandonments_per_bin)

ylim_cust   = (0, max(max(c) for c in all_customers) + 1)
ylim_abandon = (0, max(max(a) for a in all_abandon)  + 1)

for idx, (day, customers_per_bin, abandonments_per_bin) in enumerate(zip(all_dates, all_customers, all_abandon)):
    ax = axes_flat[idx]

    x = [HOUR_START + i * BIN_MIN / 60 for i in range(len(customers_per_bin))]

    line_cust, = ax.plot(x, customers_per_bin, color='steelblue', linewidth=1.5)
    ax.set_ylabel('Customers', fontsize=6, color='steelblue')
    ax.tick_params(axis='y', labelcolor='steelblue', labelsize=5)
    ax.set_xlim(HOUR_START, HOUR_END)
    ax.set_ylim(*ylim_cust)
    ax.set_xticks(range(HOUR_START, HOUR_END + 1))
    ax.set_xticklabels([str(h) for h in range(HOUR_START, HOUR_END + 1)], fontsize=5)
    ax.grid(True, alpha=0.3)
    ax.set_title(str(day), fontsize=7, fontweight='bold', pad=2)

    ax2 = ax.twinx()
    line_aband, = ax2.plot(x, abandonments_per_bin, color='tomato', linewidth=1.5, linestyle='--')
    ax2.set_ylabel('Abandonments', fontsize=6, color='tomato')
    ax2.tick_params(axis='y', labelcolor='tomato', labelsize=5)
    ax2.set_ylim(*ylim_abandon)

for idx in range(num_days, len(axes_flat)):
    axes_flat[idx].set_visible(False)

fig.legend(
    handles=[line_cust, line_aband],
    labels=['Customers in store', 'Cart abandonments'],
    loc='lower center', ncol=2, fontsize=10,
    bbox_to_anchor=(0.5, 0), bbox_transform=fig.transFigure
)

plt.suptitle('Customers in store vs cart abandonments per day (15-min bins, 15:00–16:00)',
             fontsize=12, fontweight='bold')
plt.tight_layout(rect=[0, 0.04, 1, 0.99])
plt.show()

print(f"Plotted {num_days} days")


## Number of items in abandoned carta

In [ ]:
# Basket size of customers who abandon vs complete

abandon_ids = set(log_df[log_df['concept:name'] == 'Abandon cart and leave']['case:concept:name'])

# items is the planned basket size
items_per_case = (
    log_df[log_df['concept:name'] == 'Enter store']
    .drop_duplicates('case:concept:name')
    .set_index('case:concept:name')['items']
)

abandon_items  = items_per_case[items_per_case.index.isin(abandon_ids)].dropna()
complete_items = items_per_case[~items_per_case.index.isin(abandon_ids)].dropna()

print("Abandoning customers — basket size:")
print(abandon_items.describe().round(1))
print("\nCompleting customers — basket size:")
print(complete_items.describe().round(1))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Histogram
axes[0].hist(complete_items, bins=20, color='steelblue', alpha=0.5, edgecolor='white',
             label=f'Complete (n={len(complete_items)})')
axes[0].hist(abandon_items, bins=20, color='tomato', alpha=0.7, edgecolor='white',
             label=f'Abandon (n={len(abandon_items)})')
axes[0].set_xlabel('Number of items in basket', fontsize=11)
axes[0].set_ylabel('Number of customers', fontsize=11)
axes[0].set_title('Basket size: abandon vs complete', fontsize=11)
axes[0].legend()

# Box plot
bp = axes[1].boxplot(
    [abandon_items.values, complete_items.values],
    labels=['Abandon', 'Complete'],
    patch_artist=True,
    medianprops=dict(color='black', linewidth=2)
)
for patch, color in zip(bp['boxes'], ['tomato', 'steelblue']):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)

axes[1].set_ylabel('Number of items in basket', fontsize=11)
axes[1].set_title('Basket size: abandon vs complete', fontsize=11)

plt.tight_layout()
plt.show()


## Do more price check have a correlation with higher cart abandonment?

In [ ]:
HOUR_START, HOUR_END = 14, 15
BIN_MIN = 5
NCOLS = 7

all_dates = sorted(log_df['time:timestamp'].dt.date.unique())
num_days  = len(all_dates)
tz = log_df['time:timestamp'].dt.tz

# compute data for all days
all_price  = []
all_abandon = []

for day in all_dates:
    day_data = log_df[log_df['time:timestamp'].dt.date == day]
    time_bins = pd.date_range(
        start=pd.Timestamp(str(day), tz=tz) + pd.Timedelta(hours=HOUR_START),
        end  =pd.Timestamp(str(day), tz=tz) + pd.Timedelta(hours=HOUR_END),
        freq =f'{BIN_MIN}min'
    )
    price_checks = []
    abandonments = []
    for i in range(len(time_bins) - 1):
        bin_start, bin_end = time_bins[i], time_bins[i + 1]
        price_checks.append(day_data[
            (day_data['concept:name'] == 'Start Price Check') &
            (day_data['time:timestamp'] >= bin_start) &
            (day_data['time:timestamp'] <  bin_end)
        ].shape[0])
        abandonments.append(day_data[
            (day_data['concept:name'] == 'Abandon cart and leave') &
            (day_data['time:timestamp'] >= bin_start) &
            (day_data['time:timestamp'] <  bin_end)
        ].shape[0])
    all_price.append(price_checks)
    all_abandon.append(abandonments)

# global y-limits (same for every subplot)
ylim_price  = (0, max(max(p) for p in all_price)  + 1)
ylim_abandon = (0, max(max(a) for a in all_abandon) + 1)

# plot with fixed scales
nrows = (num_days + NCOLS - 1) // NCOLS
fig, axes = plt.subplots(nrows, NCOLS, figsize=(21, 3.5 * nrows), squeeze=False)
axes_flat = axes.flatten()

line_price = None
line_aband = None

for idx, (day, price_checks, abandonments) in enumerate(zip(all_dates, all_price, all_abandon)):
    ax = axes_flat[idx]
    x = [HOUR_START + i * BIN_MIN / 60 for i in range(len(price_checks))]

    line_price, = ax.plot(x, price_checks, color='steelblue', linewidth=1.5)
    ax.set_ylabel('Price checks', fontsize=6, color='steelblue')
    ax.tick_params(axis='y', labelcolor='steelblue', labelsize=5)
    ax.set_xlim(HOUR_START, HOUR_END)
    ax.set_ylim(*ylim_price)
    ax.set_xticks(range(HOUR_START, HOUR_END + 1))
    ax.set_xticklabels([str(h) for h in range(HOUR_START, HOUR_END + 1)], fontsize=5)
    ax.grid(True, alpha=0.3)
    ax.set_title(str(day), fontsize=7, fontweight='bold', pad=2)

    ax2 = ax.twinx()
    line_aband, = ax2.plot(x, abandonments, color='tomato', linewidth=1.5, linestyle='--')
    ax2.set_ylabel('Abandonments', fontsize=6, color='tomato')
    ax2.tick_params(axis='y', labelcolor='tomato', labelsize=5)
    ax2.set_ylim(*ylim_abandon)

for idx in range(num_days, len(axes_flat)):
    axes_flat[idx].set_visible(False)

fig.legend(
    handles=[line_price, line_aband],
    labels=['Price checks', 'Cart abandonments'],
    loc='lower center', ncol=2, fontsize=10,
    bbox_to_anchor=(0.5, 0), bbox_transform=fig.transFigure
)

plt.suptitle('Price checks vs cart abandonments per day (15-min bins, 14:00–15:00)',
             fontsize=12, fontweight='bold')
plt.tight_layout(rect=[0, 0.04, 1, 0.99])
plt.show()


*************************
## **Load the clerk log**
************************

In [ ]:
clerk_log = xes_importer.apply(r"Supermarket_data\Supermarket_Clerk.xes")
clerk_df  = log_converter.apply(clerk_log, variant=log_converter.Variants.TO_DATA_FRAME)


## Analyse the schdule of clerks, the length of shifts and if certain clerks work more or less than others

How many clerks work every hour 

In [ ]:
# For each (date, hour) pair, count distinct clerks who had at least one event
clerk_df['date'] = clerk_df['time:timestamp'].dt.date
clerk_df['hour'] = clerk_df['time:timestamp'].dt.hour

clerks_per_day_hour = (
    clerk_df
    .groupby(['date', 'hour'])['case:concept:name']
    .nunique()
    .reset_index(name='n_clerks')
)

# Aggregate across days: min, max, mean per hour-of-day
hourly_stats = (
    clerks_per_day_hour
    .groupby('hour')['n_clerks']
    .agg(min='min', max='max', avg='mean')
    .reset_index()
)

print(hourly_stats.to_string(index=False))

# Plot
fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(hourly_stats['hour'], hourly_stats['avg'], label='Average', color='steelblue', alpha=0.7)
ax.errorbar(
    hourly_stats['hour'], hourly_stats['avg'],
    yerr=[hourly_stats['avg'] - hourly_stats['min'],
          hourly_stats['max'] - hourly_stats['avg']],
    fmt='none', color='black', capsize=4, label='Min / Max range'
)
ax.set_xlabel('Hour of day')
ax.set_ylabel('Number of active clerks')
ax.set_title('Clerks working per hour (min / avg / max across days)')
ax.set_xticks(hourly_stats['hour'])
ax.legend()
plt.tight_layout()
plt.show()


How long are clerk shifts 

In [ ]:
# Shift duration proxy: last event - first event per clerk per day
clerk_df['date'] = clerk_df['time:timestamp'].dt.date

shift_spans = (
    clerk_df
    .groupby(['case:concept:name', 'date'])['time:timestamp']
    .agg(shift_start='min', shift_end='max')
    .reset_index()
)

shift_spans['duration_hours'] = (
    shift_spans['shift_end'] - shift_spans['shift_start']
).dt.total_seconds() / 3600

print(shift_spans['duration_hours'].describe())

# Distribution plot
shift_spans['duration_hours'].plot(kind='hist', bins=30, edgecolor='black')
plt.xlabel('Shift duration (hours)')
plt.ylabel('Count')
plt.title('Distribution of clerk shift durations (first to last event per day)')
plt.show()


In [ ]:
print(f"Shortest shift: {shift_spans['duration_hours'].min():.2f} hours")
print(f"Longest shift:  {shift_spans['duration_hours'].max():.2f} hours")
print(f"Average shift:  {shift_spans['duration_hours'].mean():.2f} hours")


Average number of price cheks each hour per clerk 

In [ ]:
# use only Start Price Check events
pc_events = clerk_df[clerk_df['concept:name'] == 'Start Price Check'].copy()
pc_events['date'] = pc_events['time:timestamp'].dt.date
pc_events['hour'] = pc_events['time:timestamp'].dt.hour

# Count price checks per (clerk, date, hour)
pc_per_clerk_day_hour = (
    pc_events
    .groupby(['case:concept:name', 'date', 'hour'])
    .size()
    .reset_index(name='n_price_checks')
)

# Average across all (clerk, date) instances for each hour of day
avg_pc_per_hour = (
    pc_per_clerk_day_hour
    .groupby('hour')['n_price_checks']
    .mean()
)

print(avg_pc_per_hour.round(2))

avg_pc_per_hour.plot(kind='bar', color='tomato', edgecolor='black')
plt.xlabel('Hour of day')
plt.ylabel('Avg price checks per clerk')
plt.title('Average price checks per clerk per hour of day')
plt.tight_layout()
plt.show()


The average, min and max lngth of a shift for each clerk. Are there clerks who work less on average?

In [ ]:
clerk_shift_stats = (
    shift_spans
    .groupby('case:concept:name')['duration_hours']
    .agg(min='min', max='max', avg='mean')
    .round(2)
)

print(clerk_shift_stats)

# Plot
clerk_shift_stats.plot(kind='bar', figsize=(10, 5), edgecolor='black')
plt.xlabel('Clerk ID')
plt.ylabel('Shift duration (hours)')
plt.title('Shift duration per clerk (min / avg / max)')
plt.legend(['Min', 'Max', 'Avg'])
plt.tight_layout()
plt.show()


## Identify if there are clerks which are less efficient than others on average 

Number of price checks per clerk per hour of the day 

In [ ]:
pc_events = clerk_df[clerk_df['concept:name'] == 'Start Price Check'].copy()
pc_events['date'] = pc_events['time:timestamp'].dt.date
pc_events['hour'] = pc_events['time:timestamp'].dt.hour

# Count per (clerk, date, hour), then average across days
pc_per_clerk_day_hour = (
    pc_events
    .groupby(['case:concept:name', 'date', 'hour'])
    .size()
    .reset_index(name='n_price_checks')
)

avg_per_clerk_hour = (
    pc_per_clerk_day_hour
    .groupby(['hour', 'case:concept:name'])['n_price_checks']
    .mean()
    .unstack('case:concept:name')
)

print(avg_per_clerk_hour.round(2))

# Heatmap
plt.figure(figsize=(12, 6))
sns.heatmap(avg_per_clerk_hour, annot=True, fmt='.1f', cmap='YlOrRd')
plt.xlabel('Clerk ID')
plt.ylabel('Hour of day')
plt.title('Avg price checks per clerk per hour of day')
plt.tight_layout()
plt.show()


Number of cart cleanups per clerk per hour

In [ ]:
cleanup_events = clerk_df[clerk_df['concept:name'] == 'Cleanup abandoned item'].copy()
cleanup_events['date'] = cleanup_events['time:timestamp'].dt.date
cleanup_events['hour'] = cleanup_events['time:timestamp'].dt.hour

cleanup_per_clerk_day_hour = (
    cleanup_events
    .groupby(['case:concept:name', 'date', 'hour'])
    .size()
    .reset_index(name='n_cleanups')
)

avg_cleanup_per_clerk_hour = (
    cleanup_per_clerk_day_hour
    .groupby(['hour', 'case:concept:name'])['n_cleanups']
    .mean()
    .unstack('case:concept:name')
)

print(avg_cleanup_per_clerk_hour.round(2))

plt.figure(figsize=(12, 6))
sns.heatmap(avg_cleanup_per_clerk_hour, annot=True, fmt='.1f', cmap='YlOrRd')
plt.xlabel('Clerk ID')
plt.ylabel('Hour of day')
plt.title('Avg cart cleanups per clerk per hour of day')
plt.tight_layout()
plt.show()


## Cart abandonment vs cleanup vs price checks between 14:00 and 15:00 each day

In [ ]:
HOUR_START, HOUR_END = 14, 15
BIN_MIN = 5
NCOLS = 7

all_dates = sorted(log_df['time:timestamp'].dt.date.unique())
num_days  = len(all_dates)
tz = log_df['time:timestamp'].dt.tz

# precompute
all_abandon  = []
all_price    = []
all_cleanup  = []

for day in all_dates:
    day_data = log_df[log_df['time:timestamp'].dt.date == day]

    time_bins = pd.date_range(
        start=pd.Timestamp(str(day), tz=tz) + pd.Timedelta(hours=HOUR_START),
        end  =pd.Timestamp(str(day), tz=tz) + pd.Timedelta(hours=HOUR_END),
        freq =f'{BIN_MIN}min'
    )

    clerk_day = clerk_df[
        (clerk_df['time:timestamp'] >= time_bins[0]) &
        (clerk_df['time:timestamp'] <  time_bins[-1])
    ]

    abandon  = []
    price    = []
    cleanup  = []

    for i in range(len(time_bins) - 1):
        bin_start, bin_end = time_bins[i], time_bins[i + 1]

        abandon.append(day_data[
            (day_data['concept:name'] == 'Abandon cart and leave') &
            (day_data['time:timestamp'] >= bin_start) &
            (day_data['time:timestamp'] <  bin_end)
        ].shape[0])

        price.append(day_data[
            (day_data['concept:name'] == 'Start Price Check') &
            (day_data['time:timestamp'] >= bin_start) &
            (day_data['time:timestamp'] <  bin_end)
        ].shape[0])

        cleanup.append(clerk_day[
            (clerk_day['concept:name'] == 'Cleanup abandoned item') &
            (clerk_day['time:timestamp'] >= bin_start) &
            (clerk_day['time:timestamp'] <  bin_end)
        ].shape[0])

    all_abandon.append(abandon)
    all_price.append(price)
    all_cleanup.append(cleanup)

# shared y-axis scale across all days and all three metrics
global_max = max(
    max(max(a) for a in all_abandon),
    max(max(p) for p in all_price),
    max(max(c) for c in all_cleanup)
) + 1

# plot
nrows = (num_days + NCOLS - 1) // NCOLS
fig, axes = plt.subplots(nrows, NCOLS, figsize=(21, 3 * nrows), squeeze=False)
axes_flat = axes.flatten()

x = [HOUR_START + i * BIN_MIN / 60 for i in range(len(time_bins) - 1)]

for idx, (day, abandon, price, cleanup) in enumerate(
        zip(all_dates, all_abandon, all_price, all_cleanup)):

    ax = axes_flat[idx]
    ax.plot(x, abandon,  color='tomato',    linewidth=1.5, linestyle='--', label='Abandonments')
    ax.plot(x, price,    color='steelblue', linewidth=1.5, label='Price checks')
    ax.plot(x, cleanup,  color='seagreen',  linewidth=1.5, linestyle=':', label='Cleanups')

    ax.set_ylim(0, global_max)
    ax.set_xlim(HOUR_START, HOUR_END)
    ax.set_xticks([14, 14.25, 14.5, 14.75, 15])
    ax.set_xticklabels(['14:00', '14:15', '14:30', '14:45', '15:00'], fontsize=5, rotation=30)
    ax.tick_params(axis='y', labelsize=5)
    ax.set_title(str(day), fontsize=7, fontweight='bold', pad=2)
    ax.grid(True, alpha=0.3)

for idx in range(num_days, len(axes_flat)):
    axes_flat[idx].set_visible(False)

axes_flat[0].legend(fontsize=7, loc='upper left')

plt.suptitle('Cart abandonments vs price checks vs cleanups per day (5-min bins, 14:00–15:00)',
             fontsize=12, fontweight='bold')
plt.tight_layout(rect=[0, 0.01, 1, 0.99])
plt.show()

print(f"Plotted {num_days} days")


-the cleanup is per item and we cant see to which cart an item belongs 

## Clerk shift schedules


In [ ]:
clerk_df['date'] = clerk_df['time:timestamp'].dt.date

shift_bounds = (
    clerk_df.groupby(['case:concept:name', 'date'])['time:timestamp']
    .agg(shift_start='min', shift_end='max')
    .reset_index()
)
shift_bounds.columns = ['clerk', 'date', 'shift_start', 'shift_end']
shift_bounds['start_hour'] = shift_bounds['shift_start'].dt.hour + shift_bounds['shift_start'].dt.minute / 60
shift_bounds['end_hour']   = shift_bounds['shift_end'].dt.hour   + shift_bounds['shift_end'].dt.minute   / 60

all_dates = sorted(shift_bounds['date'].unique())
clerks    = sorted(shift_bounds['clerk'].unique())
num_days  = len(all_dates)
NCOLS = 7
nrows = (num_days + NCOLS - 1) // NCOLS

colors = plt.cm.Set2.colors

fig, axes = plt.subplots(nrows, NCOLS, figsize=(21, 3 * nrows), squeeze=False)
axes_flat = axes.flatten()

for idx, day in enumerate(all_dates):
    ax = axes_flat[idx]
    day_shifts = shift_bounds[shift_bounds['date'] == day]

    for y, clerk in enumerate(clerks):
        row = day_shifts[day_shifts['clerk'] == clerk]
        if len(row) == 0:
            continue
        start = row['start_hour'].values[0]
        end   = row['end_hour'].values[0]
        ax.barh(y, end - start, left=start, height=0.6,
                color=colors[y % len(colors)], alpha=0.8)

    ax.set_yticks(range(len(clerks)))
    ax.set_yticklabels([str(c) for c in clerks], fontsize=5)
    ax.set_xlim(13, 22)
    ax.set_xticks(range(13, 23))
    ax.set_xticklabels([str(h) for h in range(13, 23)], fontsize=5)
    ax.set_title(str(day), fontsize=7, fontweight='bold', pad=2)
    ax.grid(True, alpha=0.3, axis='x')

for idx in range(num_days, len(axes_flat)):
    axes_flat[idx].set_visible(False)

plt.suptitle('Clerk shift schedule per day (inferred from first/last event)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


Between 14 and 15

In [ ]:
clerk_df['date'] = clerk_df['time:timestamp'].dt.date

shift_bounds = (
    clerk_df.groupby(['case:concept:name', 'date'])['time:timestamp']
    .agg(shift_start='min', shift_end='max')
    .reset_index()
)
shift_bounds.columns = ['clerk', 'date', 'shift_start', 'shift_end']
shift_bounds['start_hour'] = shift_bounds['shift_start'].dt.hour + shift_bounds['shift_start'].dt.minute / 60
shift_bounds['end_hour']   = shift_bounds['shift_end'].dt.hour   + shift_bounds['shift_end'].dt.minute   / 60

all_dates = sorted(shift_bounds['date'].unique())
clerks    = sorted(shift_bounds['clerk'].unique())
num_days  = len(all_dates)
NCOLS = 7
nrows = (num_days + NCOLS - 1) // NCOLS

colors = plt.cm.Set2.colors

fig, axes = plt.subplots(nrows, NCOLS, figsize=(21, 3 * nrows), squeeze=False)
axes_flat = axes.flatten()

for idx, day in enumerate(all_dates):
    ax = axes_flat[idx]
    day_shifts = shift_bounds[shift_bounds['date'] == day]

    for y, clerk in enumerate(clerks):
        row = day_shifts[day_shifts['clerk'] == clerk]
        if len(row) == 0:
            continue
        start = row['start_hour'].values[0]
        end   = row['end_hour'].values[0]
        ax.barh(y, end - start, left=start, height=0.6,
                color=colors[y % len(colors)], alpha=0.8)

    ax.set_yticks(range(len(clerks)))
    ax.set_yticklabels([str(c) for c in clerks], fontsize=5)
    ax.set_xlim(14, 15)
    ax.set_xticks([14, 14.25, 14.5, 14.75, 15])
    ax.set_xticklabels(['14:00', '14:15', '14:30', '14:45', '15:00'], fontsize=5, rotation=30)
    ax.set_title(str(day), fontsize=7, fontweight='bold', pad=2)
    ax.grid(True, alpha=0.3, axis='x')

for idx in range(num_days, len(axes_flat)):
    axes_flat[idx].set_visible(False)

plt.suptitle('Clerk shift schedule per day (inferred from first/last event)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


## Price checks duration between 14 and 15

In [ ]:
# price checks are per clerk
pc_events = clerk_df[clerk_df['concept:name'].isin(['Start Price Check', 'End price check'])].copy()
pc_events = pc_events.sort_values(['case:concept:name', 'time:timestamp'])

starts = (pc_events[pc_events['concept:name'] == 'Start Price Check']
          [['case:concept:name', 'cid', 'time:timestamp']]
          .rename(columns={'time:timestamp': 'start_time'}))
ends   = (pc_events[pc_events['concept:name'] == 'End price check']
          [['case:concept:name', 'cid', 'time:timestamp']]
          .rename(columns={'time:timestamp': 'end_time'}))

starts['rank'] = starts.groupby(['case:concept:name', 'cid']).cumcount()
ends['rank']   = ends.groupby(['case:concept:name', 'cid']).cumcount()

pc_pairs = starts.merge(ends, on=['case:concept:name', 'cid', 'rank'])
pc_pairs['duration_sec'] = (pc_pairs['end_time'] - pc_pairs['start_time']).dt.total_seconds()
pc_pairs['start_hour']   = pc_pairs['start_time'].dt.hour + pc_pairs['start_time'].dt.minute / 60 + pc_pairs['start_time'].dt.second / 3600
pc_pairs['end_hour']     = pc_pairs['end_time'].dt.hour   + pc_pairs['end_time'].dt.minute   / 60 + pc_pairs['end_time'].dt.second   / 3600

window = pc_pairs[(pc_pairs['start_hour'] >= 14) & (pc_pairs['start_hour'] < 15)].copy()
window['start_min'] = (window['start_hour'] - 14) * 60
window['end_min']   = (window['end_hour']   - 14) * 60

print(f"Price checks in 14:00-15:00: {len(window)}")
print(window['duration_sec'].describe().round(1))

ffig, axes = plt.subplots(1, 3, figsize=(20, 4))

for i, (_, row) in enumerate(window.iterrows()):
    axes[0].barh(i, row['end_min'] - row['start_min'], left=row['start_min'],
                 height=1, color='steelblue', alpha=0.5)

axes[0].set_xlim(0, 60)
axes[0].set_xticks(range(0, 61, 5))
axes[0].set_xticklabels([f'14:{m:02d}' for m in range(0, 61, 5)], rotation=45, fontsize=8)
axes[0].set_xlabel('Time')
axes[0].set_ylabel('Price check (each row = one check)')
axes[0].set_title('Price check start → end (14:00–15:00, all days)')

axes[1].hist(window['duration_sec'], bins=30, color='steelblue', edgecolor='white')
axes[1].set_xlabel('Duration (seconds)')
axes[1].set_ylabel('Count')
axes[1].set_title('Price check duration distribution')

# Right: average duration per 5-min bin
window['bin'] = (window['start_min'] // 5) * 5
avg_duration = window.groupby('bin')['duration_sec'].mean()

bins = range(0, 60, 5)
axes[2].bar(
    [b + 2.5 for b in bins],
    [avg_duration.get(b, 0) for b in bins],
    width=4.5, color='steelblue', edgecolor='white'
)
axes[2].set_xticks([b + 2.5 for b in bins])
axes[2].set_xticklabels([f'14:{b:02d}' for b in bins], rotation=45, fontsize=8)
axes[2].set_xlabel('5-min bin')
axes[2].set_ylabel('Avg duration (seconds)')
axes[2].set_title('Average price check duration per 5-min bin (14:00–15:00)')
axes[2].grid(True, alpha=0.3, axis='y')


plt.tight_layout()
plt.show()


Now from 14 to 16

In [ ]:
window = pc_pairs[(pc_pairs['start_hour'] >= 14) & (pc_pairs['start_hour'] < 16)].copy()
window['start_min'] = (window['start_hour'] - 14) * 60
window['end_min']   = (window['end_hour']   - 14) * 60

fig, axes = plt.subplots(1, 3, figsize=(20, 4))

for i, (_, row) in enumerate(window.iterrows()):
    axes[0].barh(i, row['end_min'] - row['start_min'], left=row['start_min'],
                 height=1, color='steelblue', alpha=0.5)

axes[0].set_xlim(0, 120)
axes[0].set_xticks(range(0, 121, 5))
axes[0].set_xticklabels([f'{14 + m//60}:{m%60:02d}' for m in range(0, 121, 5)], rotation=45, fontsize=8)
axes[0].set_xlabel('Time')
axes[0].set_ylabel('Price check (each row = one check)')
axes[0].set_title('Price check start → end (14:00–16:00, all days)')

axes[1].hist(window['duration_sec'], bins=30, color='steelblue', edgecolor='white')
axes[1].set_xlabel('Duration (seconds)')
axes[1].set_ylabel('Count')
axes[1].set_title('Price check duration distribution')

window['bin'] = (window['start_min'] // 5) * 5
avg_duration = window.groupby('bin')['duration_sec'].mean()

bins = range(0, 120, 5)
axes[2].bar(
    [b + 2.5 for b in bins],
    [avg_duration.get(b, 0) for b in bins],
    width=4.5, color='steelblue', edgecolor='white'
)
axes[2].set_xticks([b + 2.5 for b in bins])
axes[2].set_xticklabels([f'{14 + b//60}:{b%60:02d}' for b in bins], rotation=45, fontsize=8)
axes[2].set_xlabel('5-min bin')
axes[2].set_ylabel('Avg duration (seconds)')
axes[2].set_title('Average price check duration per 5-min bin (14:00–16:00)')
axes[2].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()


In [ ]:
window = pc_pairs[(pc_pairs['start_hour'] >= 14) & (pc_pairs['start_hour'] < 15)].copy()
window['bin'] = (((window['start_hour'] - 14) * 60) // 5) * 5

bins = range(0, 60, 5)
labels = [f'14:{b:02d}' for b in bins]

groups = [window[window['bin'] == b]['duration_sec'].values for b in bins]
counts = [len(g) for g in groups]

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True,
                         gridspec_kw={'height_ratios': [3, 1]})

# Top: box plot
axes[0].boxplot(
    groups, positions=range(len(labels)), labels=labels,
    patch_artist=True,
    boxprops=dict(facecolor='steelblue', alpha=0.6),
    medianprops=dict(color='black', linewidth=2),
    flierprops=dict(marker='o', markersize=3, alpha=0.3)
)
axes[0].set_ylabel('Duration (seconds)')
axes[0].set_title('Price check duration per 5-min bin (14:00–15:00)')
axes[0].grid(True, alpha=0.3, axis='y')
axes[0].tick_params(axis='x', rotation=45)

# Bottom: count bar
axes[1].bar(labels, counts, color='steelblue', alpha=0.7, edgecolor='white')
for i, (label, count) in enumerate(zip(labels, counts)):
    axes[1].text(i, count + 0.1, str(count), ha='center', va='bottom', fontsize=8)
axes[1].set_ylabel('Count')
axes[1].set_xlabel('5-min bin')
axes[1].grid(True, alpha=0.3, axis='y')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()


Average amount of time a clerk spends on price checks. We take into account how many are working 

In [ ]:
pc_pairs['date'] = pc_pairs['start_time'].dt.date
tz = clerk_df['time:timestamp'].dt.tz

bin_records = []

for day in sorted(pc_pairs['date'].unique()):
    day_str = str(day)
    day_start = pd.Timestamp(day_str, tz=tz)

    time_bins = pd.date_range(
        start=day_start + pd.Timedelta(hours=14),
        end  =day_start + pd.Timedelta(hours=15),
        freq ='5min'
    )

    # Each clerk's shift bounds on this day (first and last event)
    clerk_today = clerk_df[
        (clerk_df['time:timestamp'] >= day_start) &
        (clerk_df['time:timestamp'] <  day_start + pd.Timedelta(days=1))
    ]
    if len(clerk_today) == 0:
        continue

    clerk_shifts = clerk_today.groupby('case:concept:name')['time:timestamp'].agg(
        shift_start='min', shift_end='max'
    )

    day_pairs = pc_pairs[
        (pc_pairs['date'] == day) &
        (pc_pairs['start_hour'] >= 14) &
        (pc_pairs['start_hour'] <  15)
    ]

    for i in range(len(time_bins) - 1):
        bin_start, bin_end = time_bins[i], time_bins[i + 1]

        # Only count clerks whose shift overlaps this specific bin
        n_clerks = clerk_shifts[
            (clerk_shifts['shift_start'] < bin_end) &
            (clerk_shifts['shift_end']   > bin_start)
        ].shape[0]

        if n_clerks == 0:
            continue

        in_bin = day_pairs[
            (day_pairs['start_time'] < bin_end) &
            (day_pairs['end_time']   > bin_start)
        ]
        bin_records.append({
            'bin_min':       i * 5,
            'total_pc_sec':  in_bin['duration_sec'].sum(),
            'n_clerks':      n_clerks,
            'avg_per_clerk': in_bin['duration_sec'].sum() / n_clerks
        })

bin_df = pd.DataFrame(bin_records)
avg_by_bin = bin_df.groupby('bin_min').agg(
    avg_per_clerk=('avg_per_clerk', 'mean'),
    avg_n_clerks =('n_clerks',      'mean')
).reset_index()

labels = [f'{14 + b//60}:{b%60:02d}' for b in avg_by_bin['bin_min']]

fig, axes = plt.subplots(2, 1, figsize=(18, 7), sharex=True,
                         gridspec_kw={'height_ratios': [3, 1]})

axes[0].bar(labels, avg_by_bin['avg_per_clerk'], color='steelblue', alpha=0.7, edgecolor='white')
for i, val in enumerate(avg_by_bin['avg_per_clerk']):
    axes[0].text(i, val + 0.3, f'{val:.1f}s', ha='center', va='bottom', fontsize=7)
axes[0].set_ylabel('Avg seconds on price checks per clerk')
axes[0].set_title('Average time per clerk spent on price checks per 5-min bin (14:00–15:00)')
axes[0].grid(True, alpha=0.3, axis='y')

axes[1].bar(labels, avg_by_bin['avg_n_clerks'], color='seagreen', alpha=0.7, edgecolor='white')
for i, val in enumerate(avg_by_bin['avg_n_clerks']):
    axes[1].text(i, val + 0.05, f'{val:.1f}', ha='center', va='bottom', fontsize=7)
axes[1].set_ylabel('Avg clerks working')
axes[1].set_xlabel('5-min bin')
axes[1].tick_params(axis='x', rotation=45, labelsize=7)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()


this is the total amount of time a clerk (one) spends doing price checks at a moment in a day. we take into account how many clerks are working at that time. 

In [ ]:
pc_pairs['date'] = pc_pairs['start_time'].dt.date
tz = clerk_df['time:timestamp'].dt.tz

bin_records = []

for day in sorted(pc_pairs['date'].unique()):
    day_str = str(day)
    day_start = pd.Timestamp(day_str, tz=tz)

    time_bins = pd.date_range(
        start=day_start + pd.Timedelta(hours=14),
        end  =day_start + pd.Timedelta(hours=17),
        freq ='5min'
    )

    # Each clerk's shift bounds on this day (first and last event)
    clerk_today = clerk_df[
        (clerk_df['time:timestamp'] >= day_start) &
        (clerk_df['time:timestamp'] <  day_start + pd.Timedelta(days=1))
    ]
    if len(clerk_today) == 0:
        continue

    clerk_shifts = clerk_today.groupby('case:concept:name')['time:timestamp'].agg(
        shift_start='min', shift_end='max'
    )

    day_pairs = pc_pairs[
        (pc_pairs['date'] == day) &
        (pc_pairs['start_hour'] >= 14) &
        (pc_pairs['start_hour'] <  17)
    ]

    for i in range(len(time_bins) - 1):
        bin_start, bin_end = time_bins[i], time_bins[i + 1]

        # Only count clerks whose shift overlaps this specific bin
        n_clerks = clerk_shifts[
            (clerk_shifts['shift_start'] < bin_end) &
            (clerk_shifts['shift_end']   > bin_start)
        ].shape[0]

        if n_clerks == 0:
            continue

        in_bin = day_pairs[
            (day_pairs['start_time'] < bin_end) &
            (day_pairs['end_time']   > bin_start)
        ]
        bin_records.append({
            'bin_min':       i * 5,
            'total_pc_sec':  in_bin['duration_sec'].sum(),
            'n_clerks':      n_clerks,
            'avg_per_clerk': in_bin['duration_sec'].sum() / n_clerks
        })

bin_df = pd.DataFrame(bin_records)
avg_by_bin = bin_df.groupby('bin_min').agg(
    avg_per_clerk=('avg_per_clerk', 'mean'),
    avg_n_clerks =('n_clerks',      'mean')
).reset_index()

labels = [f'{14 + b//60}:{b%60:02d}' for b in avg_by_bin['bin_min']]

fig, axes = plt.subplots(2, 1, figsize=(18, 7), sharex=True,
                         gridspec_kw={'height_ratios': [3, 1]})

axes[0].bar(labels, avg_by_bin['avg_per_clerk'], color='steelblue', alpha=0.7, edgecolor='white')
for i, val in enumerate(avg_by_bin['avg_per_clerk']):
    axes[0].text(i, val + 0.3, f'{val:.1f}s', ha='center', va='bottom', fontsize=7)
axes[0].set_ylabel('Avg seconds on price checks per clerk')
axes[0].set_title('Average time per clerk spent on price checks per 5-min bin (14:00–17:00)')
axes[0].grid(True, alpha=0.3, axis='y')

axes[1].bar(labels, avg_by_bin['avg_n_clerks'], color='seagreen', alpha=0.7, edgecolor='white')
for i, val in enumerate(avg_by_bin['avg_n_clerks']):
    axes[1].text(i, val + 0.05, f'{val:.1f}', ha='center', va='bottom', fontsize=7)
axes[1].set_ylabel('Avg clerks working')
axes[1].set_xlabel('5-min bin')
axes[1].tick_params(axis='x', rotation=45, labelsize=7)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()


In [ ]:
pc_pairs['date'] = pc_pairs['start_time'].dt.date
tz = clerk_df['time:timestamp'].dt.tz

bin_records = []

for day in sorted(pc_pairs['date'].unique()):
    day_str = str(day)
    day_start = pd.Timestamp(day_str, tz=tz)

    time_bins = pd.date_range(
        start=day_start + pd.Timedelta(hours=14),
        end  =day_start + pd.Timedelta(hours=22),
        freq ='5min'
    )

    # Each clerk's shift bounds on this day (first and last event)
    clerk_today = clerk_df[
        (clerk_df['time:timestamp'] >= day_start) &
        (clerk_df['time:timestamp'] <  day_start + pd.Timedelta(days=1))
    ]
    if len(clerk_today) == 0:
        continue

    clerk_shifts = clerk_today.groupby('case:concept:name')['time:timestamp'].agg(
        shift_start='min', shift_end='max'
    )

    day_pairs = pc_pairs[
        (pc_pairs['date'] == day) &
        (pc_pairs['start_hour'] >= 14) &
        (pc_pairs['start_hour'] <  22)
    ]

    for i in range(len(time_bins) - 1):
        bin_start, bin_end = time_bins[i], time_bins[i + 1]

        # Only count clerks whose shift overlaps this specific bin
        n_clerks = clerk_shifts[
            (clerk_shifts['shift_start'] < bin_end) &
            (clerk_shifts['shift_end']   > bin_start)
        ].shape[0]

        if n_clerks == 0:
            continue

        in_bin = day_pairs[
            (day_pairs['start_time'] < bin_end) &
            (day_pairs['end_time']   > bin_start)
        ]
        bin_records.append({
            'bin_min':       i * 5,
            'total_pc_sec':  in_bin['duration_sec'].sum(),
            'n_clerks':      n_clerks,
            'avg_per_clerk': in_bin['duration_sec'].sum() / n_clerks
        })

bin_df = pd.DataFrame(bin_records)
avg_by_bin = bin_df.groupby('bin_min').agg(
    avg_per_clerk=('avg_per_clerk', 'mean'),
    avg_n_clerks =('n_clerks',      'mean')
).reset_index()

labels = [f'{14 + b//60}:{b%60:02d}' for b in avg_by_bin['bin_min']]

fig, axes = plt.subplots(2, 1, figsize=(18, 7), sharex=True,
                         gridspec_kw={'height_ratios': [3, 1]})

axes[0].bar(labels, avg_by_bin['avg_per_clerk'], color='steelblue', alpha=0.7, edgecolor='white')
for i, val in enumerate(avg_by_bin['avg_per_clerk']):
    axes[0].text(i, val + 0.3, f'{val:.1f}s', ha='center', va='bottom', fontsize=7)
axes[0].set_ylabel('Avg seconds on price checks per clerk')
axes[0].set_title('Average time per clerk spent on price checks per 5-min bin (14:00–22:00)')
axes[0].grid(True, alpha=0.3, axis='y')

axes[1].bar(labels, avg_by_bin['avg_n_clerks'], color='seagreen', alpha=0.7, edgecolor='white')
for i, val in enumerate(avg_by_bin['avg_n_clerks']):
    axes[1].text(i, val + 0.05, f'{val:.1f}', ha='center', va='bottom', fontsize=7)
axes[1].set_ylabel('Avg clerks working')
axes[1].set_xlabel('5-min bin')
axes[1].tick_params(axis='x', rotation=45, labelsize=7)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()


## Number of customers in queue every 5 minutes looking a the counter thay choose

In [ ]:
eq = log_df[
    (log_df['concept:name'] == 'Enter Queue') &
    (log_df['time:timestamp'].dt.hour == 14)
].copy()

eq['bin_min'] = (eq['time:timestamp'].dt.minute // 5) * 5

by_bin_c = eq.groupby('bin_min')['mC'].agg(['mean', 'std', 'count'])
by_bin_i = eq.groupby('bin_min')['mI'].agg(['mean', 'std', 'count'])

bins   = sorted(eq['bin_min'].unique())
labels = [f'14:{b:02d}' for b in bins]
x      = range(len(bins))

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

# Top: customers in queue
axes[0].bar(x, by_bin_c['mean'], width=0.6, color='steelblue', alpha=0.7, edgecolor='white')
axes[0].errorbar(x, by_bin_c['mean'], yerr=by_bin_c['std'],
                 fmt='none', color='black', capsize=4, linewidth=1)
for i, (mean, count) in enumerate(zip(by_bin_c['mean'], by_bin_c['count'])):
    axes[0].text(i, mean + by_bin_c['std'].iloc[i] + 0.05,
                 f'{mean:.1f}\n(n={count})', ha='center', va='bottom', fontsize=7)
axes[0].set_ylabel('Avg customers in queue')
axes[0].set_title('Queue length per 5-min bin (14:00–15:00, avg ± std across all days)')
axes[0].grid(True, alpha=0.3, axis='y')

# Bottom: items in queue
axes[1].bar(x, by_bin_i['mean'], width=0.6, color='tomato', alpha=0.7, edgecolor='white')
axes[1].errorbar(x, by_bin_i['mean'], yerr=by_bin_i['std'],
                 fmt='none', color='black', capsize=4, linewidth=1)
for i, (mean, count) in enumerate(zip(by_bin_i['mean'], by_bin_i['count'])):
    axes[1].text(i, mean + by_bin_i['std'].iloc[i] + 0.1,
                 f'{mean:.1f}', ha='center', va='bottom', fontsize=7)
axes[1].set_ylabel('Avg items in queue')
axes[1].set_xlabel('5-min bin')
axes[1].set_xticks(list(x))
axes[1].set_xticklabels(labels, rotation=45)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()


In [ ]:
eq = log_df[log_df['concept:name'] == 'Enter Queue'].copy()

eq['bin_min'] = (
    eq['time:timestamp'].dt.hour * 60 +
    (eq['time:timestamp'].dt.minute // 5) * 5
)

by_bin_c = eq.groupby('bin_min')['mC'].agg(['mean', 'std', 'count'])
by_bin_i = eq.groupby('bin_min')['mI'].agg(['mean', 'std', 'count'])

bins   = sorted(eq['bin_min'].unique())
labels = [f'{b//60}:{b%60:02d}' for b in bins]
x      = range(len(bins))

fig, axes = plt.subplots(2, 1, figsize=(16, 7), sharex=True)

axes[0].bar(x, by_bin_c['mean'], width=0.6, color='steelblue', alpha=0.7, edgecolor='white')
axes[0].errorbar(x, by_bin_c['mean'], yerr=by_bin_c['std'],
                 fmt='none', color='black', capsize=4, linewidth=1)
for i, (mean, count) in enumerate(zip(by_bin_c['mean'], by_bin_c['count'])):
    axes[0].text(i, mean + by_bin_c['std'].iloc[i] + 0.05,
                 f'{mean:.1f}\n(n={count})', ha='center', va='bottom', fontsize=6)
axes[0].set_ylabel('Avg customers in queue')
axes[0].set_title('Queue length per 5-min bin (full day, avg ± std across all days)')
axes[0].grid(True, alpha=0.3, axis='y')

axes[1].bar(x, by_bin_i['mean'], width=0.6, color='tomato', alpha=0.7, edgecolor='white')
axes[1].errorbar(x, by_bin_i['mean'], yerr=by_bin_i['std'],
                 fmt='none', color='black', capsize=4, linewidth=1)
for i, mean in enumerate(by_bin_i['mean']):
    axes[1].text(i, mean + by_bin_i['std'].iloc[i] + 0.1,
                 f'{mean:.1f}', ha='center', va='bottom', fontsize=6)
axes[1].set_ylabel('Avg items in queue')
axes[1].set_xlabel('Time of day')
axes[1].set_xticks(list(x))
axes[1].set_xticklabels(labels, rotation=45, fontsize=7)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()


## Average  number of customers in queue every 5 minues

In [ ]:
# Total queue depth across all open counters, derived from the col attribute
eq = log_df[log_df['concept:name'] == 'Enter Queue'].dropna(subset=['col']).copy()

eq['total_queue_customers'] = eq['col'].apply(
    lambda c: sum(x[1] for x in parse_col(c) if x[3])
)
eq['total_queue_items'] = eq['col'].apply(
    lambda c: sum(x[2] for x in parse_col(c) if x[3])
)

eq['bin_min'] = (
    eq['time:timestamp'].dt.hour * 60 +
    (eq['time:timestamp'].dt.minute // 5) * 5
)

by_bin_c = eq.groupby('bin_min')['total_queue_customers'].agg(['mean', 'std', 'count'])
by_bin_i = eq.groupby('bin_min')['total_queue_items'].agg(['mean', 'std', 'count'])

bins   = sorted(eq['bin_min'].unique())
labels = [f'{b//60}:{b%60:02d}' for b in bins]
x      = range(len(bins))

fig, axes = plt.subplots(2, 1, figsize=(16, 7), sharex=True)

axes[0].bar(x, by_bin_c['mean'], width=0.6, color='steelblue', alpha=0.7, edgecolor='white')
axes[0].errorbar(x, by_bin_c['mean'], yerr=by_bin_c['std'],
                 fmt='none', color='black', capsize=4, linewidth=1)
for i, (mean, count) in enumerate(zip(by_bin_c['mean'], by_bin_c['count'])):
    axes[0].text(i, mean + by_bin_c['std'].iloc[i] + 0.05,
                 f'{mean:.1f}\n(n={int(count)})', ha='center', va='bottom', fontsize=6)
axes[0].set_ylabel('Avg total customers in queue\n(all open counters)')
axes[0].set_title('Total queue depth per 5-min bin (full day, avg ± std across all days)')
axes[0].grid(True, alpha=0.3, axis='y')

axes[1].bar(x, by_bin_i['mean'], width=0.6, color='tomato', alpha=0.7, edgecolor='white')
axes[1].errorbar(x, by_bin_i['mean'], yerr=by_bin_i['std'],
                 fmt='none', color='black', capsize=4, linewidth=1)
for i, (mean, count) in enumerate(zip(by_bin_i['mean'], by_bin_i['count'])):
    axes[1].text(i, mean + by_bin_i['std'].iloc[i] + 0.5,
                 f'{mean:.1f}\n(n={int(count)})', ha='center', va='bottom', fontsize=6)
axes[1].set_ylabel('Avg total items in queue\n(all open counters)')
axes[1].set_xlabel('Time of day')
axes[1].set_xticks(list(x))
axes[1].set_xticklabels(labels, rotation=45, fontsize=7)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()


Price check vs nr of customers vs cart abandonment between 14 and 15

## Do price cheks slow down down the process in the couter?

In [ ]:
# Total queue depth across all open counters, derived from the col attribute
eq = log_df[log_df['concept:name'] == 'Enter Queue'].dropna(subset=['col']).copy()

eq['total_queue_customers'] = eq['col'].apply(
    lambda c: sum(x[1] for x in parse_col(c) if x[3])
)
eq['total_queue_items'] = eq['col'].apply(
    lambda c: sum(x[2] for x in parse_col(c) if x[3])
)

eq['bin_min'] = (
    eq['time:timestamp'].dt.hour * 60 +
    (eq['time:timestamp'].dt.minute // 5) * 5
)

by_bin_c = eq.groupby('bin_min')['total_queue_customers'].agg(['mean', 'std', 'count'])
by_bin_i = eq.groupby('bin_min')['total_queue_items'].agg(['mean', 'std', 'count'])

bins   = sorted(eq['bin_min'].unique())
labels = [f'{b//60}:{b%60:02d}' for b in bins]
x      = range(len(bins))

fig, axes = plt.subplots(2, 1, figsize=(16, 7), sharex=True)

axes[0].bar(x, by_bin_c['mean'], width=0.6, color='steelblue', alpha=0.7, edgecolor='white')
axes[0].errorbar(x, by_bin_c['mean'], yerr=by_bin_c['std'],
                 fmt='none', color='black', capsize=4, linewidth=1)
for i, (mean, count) in enumerate(zip(by_bin_c['mean'], by_bin_c['count'])):
    axes[0].text(i, mean + by_bin_c['std'].iloc[i] + 0.05,
                 f'{mean:.1f}\n(n={int(count)})', ha='center', va='bottom', fontsize=6)
axes[0].set_ylabel('Avg total customers in queue\n(all open counters)')
axes[0].set_title('Total queue depth per 5-min bin (full day, avg ± std across all days)')
axes[0].grid(True, alpha=0.3, axis='y')

axes[1].bar(x, by_bin_i['mean'], width=0.6, color='tomato', alpha=0.7, edgecolor='white')
axes[1].errorbar(x, by_bin_i['mean'], yerr=by_bin_i['std'],
                 fmt='none', color='black', capsize=4, linewidth=1)
for i, (mean, count) in enumerate(zip(by_bin_i['mean'], by_bin_i['count'])):
    axes[1].text(i, mean + by_bin_i['std'].iloc[i] + 0.5,
                 f'{mean:.1f}\n(n={int(count)})', ha='center', va='bottom', fontsize=6)
axes[1].set_ylabel('Avg total items in queue\n(all open counters)')
axes[1].set_xlabel('Time of day')
axes[1].set_xticks(list(x))
axes[1].set_xticklabels(labels, rotation=45, fontsize=7)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()


****************************************
## Load counter and cashier log 
****************************************

In [ ]:
counter_log = xes_importer.apply(r"Supermarket_data\Supermarket_Counter.xes")
counter_df  = log_converter.apply(counter_log, variant=log_converter.Variants.TO_DATA_FRAME)


In [ ]:
# Update this path to your XES file
new_path = "Supermarket_data\Supermarket_Cashier.xes"

log_cashier = xes_importer.apply(new_path)

print(type(log_cashier))
print(f"Number of traces: {len(log_cashier)}")


log_df_cashier = log_converter.apply(log_cashier, variant=log_converter.Variants.TO_DATA_FRAME)

log_df_cashier.head()


Check missing values

In [ ]:
starts = log_df_cashier[log_df_cashier['concept:name'] == 'Start Shift'][['case:concept:name', 'time:timestamp']].reset_index(drop=True)
ends = log_df_cashier[log_df_cashier['concept:name'] == 'End shift'][['case:concept:name', 'time:timestamp']].reset_index(drop=True)

# pair them by order (assuming 1st start goes with 1st end per cashier etc.)
shifts = pd.DataFrame({
    'case:concept:name': starts['case:concept:name'],
    'shift_start': starts['time:timestamp'],
    'shift_end': ends['time:timestamp']
})

# check if start and end are on the same day
shifts['same_day'] = shifts['shift_start'].dt.date == shifts['shift_end'].dt.date
shifts['duration_hours'] = (shifts['shift_end'] - shifts['shift_start']).dt.total_seconds() / 3600

# how many shifts cross midnight?
print(f"Shifts crossing midnight: {(~shifts['same_day']).sum()}")
print(f"Shifts where end < start: {(shifts['duration_hours'] < 0).sum()}")
print()
print(shifts['duration_hours'].describe())


In [ ]:
from scipy import stats

# forward fill caid within each counter so scan items know which cashier did them
counter = counter_df.sort_values(['case:concept:name', 'time:timestamp']).copy()
counter['caid'] = counter.groupby('case:concept:name')['caid'].ffill()

# drop customers if they don't have a valid cashier assignment
customer_items = (
    counter[counter['concept:name'] == 'Enter Queue']
    [['id', 'items', 'case:concept:name']]
    .dropna(subset=['id'])
    .drop_duplicates(['id', 'case:concept:name'])
)

# scanning duration per customer = first scan item to start payment
def session_stats(g):
    scans = g[g['concept:name'] == 'Scan Item']
    payment = g[g['concept:name'] == 'Start Payment']
    if scans.empty or payment.empty:
        return None
    return pd.Series({
        'first_scan_ts': scans['time:timestamp'].min(),
        'payment_ts':    payment['time:timestamp'].min(),
        'caid':          scans['caid'].iloc[0],
    })

sessions = (
    counter[counter['concept:name'].isin(['Scan Item', 'Start Payment'])]
    .groupby(['case:concept:name', 'id'])
    .apply(session_stats)
    .dropna()
    .reset_index()
)

sessions['scan_duration_sec'] = (sessions['payment_ts'] - sessions['first_scan_ts']).dt.total_seconds()
sessions = sessions.merge(customer_items, on=['id', 'case:concept:name'], how='left')
sessions['sec_per_item'] = sessions['scan_duration_sec'] / sessions['items']

# match each session to the cashier's most recent shift start
shift_starts = starts.rename(columns={'case:concept:name': 'caid', 'time:timestamp': 'shift_start'})
shift_starts['caid'] = shift_starts['caid'].astype(float)

# match each session to the cashier's most recent shift start
merged = pd.merge_asof(
    sessions.sort_values('first_scan_ts'),
    shift_starts.sort_values('shift_start'),
    left_on='first_scan_ts', right_on='shift_start',
    by='caid', direction='backward'
)

merged['time_into_shift_h'] = (merged['first_scan_ts'] - merged['shift_start']).dt.total_seconds() / 3600
merged = merged.dropna(subset=['time_into_shift_h', 'sec_per_item'])

# plot
slope, intercept, r, p, _ = stats.linregress(merged['time_into_shift_h'], merged['sec_per_item'])

plt.figure(figsize=(10, 5))
plt.scatter(merged['time_into_shift_h'], merged['sec_per_item'], alpha=0.2, s=5)
x = [merged['time_into_shift_h'].min(), merged['time_into_shift_h'].max()]
plt.plot(x, [slope * v + intercept for v in x], color='red', label=f'r={r:.2f}, p={p:.3f}')
plt.xlabel('Time into shift (hours)')
plt.ylabel('Seconds per item scanned')
plt.title('Does cashier scanning speed decrease over a shift?')
plt.legend()
plt.tight_layout()
plt.show()


## Scanning speed per cashier

In [ ]:
results = []

for counter_id, grp in counter_df.groupby('case:concept:name'):
    grp = grp.sort_values('time:timestamp').copy()
    grp['caid'] = grp['caid'].ffill()

    # each Enter Queue starts a new customer session
    grp['session'] = (grp['concept:name'] == 'Enter Queue').cumsum()
    grp = grp[grp['session'] > 0]

    for _, sess in grp.groupby('session'):
        scans   = sess[sess['concept:name'] == 'Scan Item']
        payment = sess[sess['concept:name'] == 'Start Payment']

        if scans.empty or payment.empty:
            continue

        caid_vals = sess['caid'].dropna()
        if caid_vals.empty:
            continue

        first_scan    = scans['time:timestamp'].min()
        pay_time      = payment['time:timestamp'].min()
        scan_duration = (pay_time - first_scan).total_seconds()
        n_items       = len(scans)

        if scan_duration <= 0 or n_items == 0:
            continue

        results.append({
            'caid':             caid_vals.iloc[0],
            'scan_duration_sec': scan_duration,
            'n_items':          n_items,
            'sec_per_item':     scan_duration / n_items,
        })

scan_df = pd.DataFrame(results)

# ── stats per cashier ─────────────────────────────────────────────────────────
per_cashier = (
    scan_df.groupby('caid')['sec_per_item']
    .agg(['min', 'mean', 'max', 'count'])
    .round(2)
    .sort_values('mean')
)
print(per_cashier)

# ── plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 5))

x = range(len(per_cashier))
ax.vlines(x, per_cashier['min'], per_cashier['max'], color='steelblue', linewidth=2)
ax.scatter(x, per_cashier['mean'], color='tomato', zorder=3, label='mean')
ax.scatter(x, per_cashier['min'],  color='steelblue', marker='_', s=100, zorder=3)
ax.scatter(x, per_cashier['max'],  color='steelblue', marker='_', s=100, zorder=3)
ax.axhline(per_cashier['mean'].mean(), color='red', linestyle='--',
           label=f"overall mean = {per_cashier['mean'].mean():.2f}s")

ax.set_xticks(list(x))
ax.set_xticklabels(per_cashier.index.astype(str), rotation=90)
ax.set_xlabel('Cashier ID')
ax.set_ylabel('Seconds per item')
ax.set_title('Scanning speed per cashier — min, mean, max')
ax.legend()
plt.tight_layout()
plt.show()


Substracting price checks 

In [ ]:
results = []

for counter_id, grp in counter_df.groupby('case:concept:name'):
    grp = grp.sort_values('time:timestamp').copy()
    grp['caid'] = grp['caid'].ffill()
    grp['session'] = (grp['concept:name'] == 'Enter Queue').cumsum()
    grp = grp[grp['session'] > 0]

    for _, sess in grp.groupby('session'):
        scans   = sess[sess['concept:name'] == 'Scan Item']
        payment = sess[sess['concept:name'] == 'Start Payment']

        if scans.empty or payment.empty:
            continue

        caid_vals = sess['caid'].dropna()
        if caid_vals.empty:
            continue

        first_scan    = scans['time:timestamp'].min()
        pay_time      = payment['time:timestamp'].min()
        scan_duration = (pay_time - first_scan).total_seconds()

        pc_starts = sess[sess['concept:name'] == 'Start Price Check']['time:timestamp'].sort_values().reset_index(drop=True)
        pc_ends   = sess[sess['concept:name'] == 'End price check']['time:timestamp'].sort_values().reset_index(drop=True)
        n_pc = min(len(pc_starts), len(pc_ends))
        for i in range(n_pc):
            scan_duration -= (pc_ends.iloc[i] - pc_starts.iloc[i]).total_seconds()

        n_items = len(scans)
        if scan_duration <= 0 or n_items == 0:
            continue

        results.append({
            'caid':         caid_vals.iloc[0],
            'sec_per_item': scan_duration / n_items,
        })

scan_df = pd.DataFrame(results)

per_cashier = (
    scan_df.groupby('caid')['sec_per_item']
    .agg(['min', 'mean', 'max', 'count'])
    .round(2)
    .sort_values('mean')
)
print(per_cashier)

fig, ax = plt.subplots(figsize=(13, 5))
x = range(len(per_cashier))
ax.vlines(x, per_cashier['min'], per_cashier['max'], color='steelblue', linewidth=2)
ax.scatter(x, per_cashier['mean'], color='tomato', zorder=3, label='mean')
ax.scatter(x, per_cashier['min'],  color='steelblue', marker='_', s=100, zorder=3)
ax.scatter(x, per_cashier['max'],  color='steelblue', marker='_', s=100, zorder=3)
ax.axhline(per_cashier['mean'].mean(), color='red', linestyle='--',
           label=f"overall mean = {per_cashier['mean'].mean():.2f}s")
ax.set_xticks(list(x))
ax.set_xticklabels(per_cashier.index.astype(str), rotation=90)
ax.set_xlabel('Cashier ID')
ax.set_ylabel('Seconds per item')
ax.set_title('Scanning speed per cashier — min, mean, max (price check time excluded)')
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
scan_df[scan_df['sec_per_item'] > 20].sort_values('sec_per_item', ascending=False)


In [ ]:
per_cashier = (
    scan_df[scan_df['sec_per_item'] < 15]
    .groupby('caid')['sec_per_item']
    .agg(['min', 'mean', 'max', 'count'])
    .round(2)
    .sort_values('mean')
)
print(per_cashier)

fig, ax = plt.subplots(figsize=(13, 5))
x = range(len(per_cashier))
ax.vlines(x, per_cashier['min'], per_cashier['max'], color='steelblue', linewidth=2)
ax.scatter(x, per_cashier['mean'], color='tomato', zorder=3, label='mean')
ax.scatter(x, per_cashier['min'],  color='steelblue', marker='_', s=100, zorder=3)
ax.scatter(x, per_cashier['max'],  color='steelblue', marker='_', s=100, zorder=3)
ax.axhline(per_cashier['mean'].mean(), color='red', linestyle='--',
           label=f"overall mean = {per_cashier['mean'].mean():.2f}s")
ax.set_xticks(list(x))
ax.set_xticklabels(per_cashier.index.astype(str), rotation=90)
ax.set_xlabel('Cashier ID')
ax.set_ylabel('Seconds per item')
ax.set_title('Scanning speed per cashier — min, mean, max (price check time excluded)')
ax.legend()
plt.tight_layout()
plt.show()


## How does cashier availability influence waiting time

In [ ]:
# compute waiting time per customer = time from Enter Queue to first Scan Item
enter_queue = counter_df[counter_df['concept:name'] == 'Enter Queue'].copy()
enter_queue = enter_queue[['case:concept:name', 'id', 'time:timestamp']]
enter_queue = enter_queue.rename(columns={'time:timestamp': 'queue_ts'})

# get the first scan item per customer per counter
first_scan = counter_df[counter_df['concept:name'] == 'Scan Item'].copy()
first_scan = first_scan.groupby(['case:concept:name', 'id'])['time:timestamp'].min()
first_scan = first_scan.reset_index()
first_scan = first_scan.rename(columns={'time:timestamp': 'first_scan_ts'})

# merge to pair each queue entry with its first scan
wait_df = enter_queue.merge(first_scan, on=['case:concept:name', 'id'])
wait_df['wait_min'] = (wait_df['first_scan_ts'] - wait_df['queue_ts']).dt.total_seconds() / 60

# remove negatives (simulation artifacts)
wait_df = wait_df[wait_df['wait_min'] >= 0]
wait_df['hour'] = wait_df['queue_ts'].dt.hour

avg_wait_per_hour = wait_df.groupby('hour')['wait_min'].mean()
print(avg_wait_per_hour)


In [ ]:
# compute active cashiers per hour from the cashier log
cashier_events = log_df_cashier[log_df_cashier['concept:name'].isin(['Start Shift', 'End shift'])].copy()
cashier_events = cashier_events.sort_values('time:timestamp')

# add +1 for shift start and -1 for shift end to track active cashiers over time
cashier_events['change'] = 0
cashier_events.loc[cashier_events['concept:name'] == 'Start Shift', 'change'] = 1
cashier_events.loc[cashier_events['concept:name'] == 'End shift', 'change'] = -1
cashier_events['active_cashiers'] = cashier_events['change'].cumsum()

cashier_events['hour'] = cashier_events['time:timestamp'].dt.hour
active_per_hour = cashier_events.groupby('hour')['active_cashiers'].mean()
print(active_per_hour)


In [ ]:
# plot waiting time and active cashiers side by side
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.plot(avg_wait_per_hour.index, avg_wait_per_hour.values, color='tomato', marker='o')
ax1.set_xlabel('Hour of day')
ax1.set_ylabel('Avg waiting time (minutes)')
ax1.set_title('Average waiting time per hour')

ax2.plot(active_per_hour.index, active_per_hour.values, color='steelblue', marker='o')
ax2.set_xlabel('Hour of day')
ax2.set_ylabel('Active cashiers')
ax2.set_title('Active cashiers per hour')

plt.tight_layout()
plt.show()


Number of price checks and number of counters open

In [ ]:
# get number of open counters per hour from the cashier log
counter_events = log_df_cashier[log_df_cashier['concept:name'].isin(['Open counter', 'Close counter'])].copy()
counter_events = counter_events.sort_values('time:timestamp')

counter_events['change'] = 0
counter_events.loc[counter_events['concept:name'] == 'Open counter', 'change'] = 1
counter_events.loc[counter_events['concept:name'] == 'Close counter', 'change'] = -1
counter_events['active_counters'] = counter_events['change'].cumsum()

counter_events['hour'] = counter_events['time:timestamp'].dt.hour
active_counters_per_hour = counter_events.groupby('hour')['active_counters'].mean()
print(active_counters_per_hour)


In [ ]:
# plot price checks vs active counters per hour
price_checks = clerk_df[clerk_df['concept:name'] == 'Start Price Check'].copy()
price_checks['hour'] = price_checks['time:timestamp'].dt.hour
pc_per_hour = price_checks.groupby('hour').size()
fig, ax1 = plt.subplots(figsize=(10, 5))

ax2 = ax1.twinx()
ax1.bar(pc_per_hour.index, pc_per_hour.values, color='tomato', alpha=0.6, label='price checks')
ax2.plot(active_counters_per_hour.index, active_counters_per_hour.values, color='steelblue', marker='o', label='active counters')

ax1.set_xlabel('Hour of day')
ax1.set_ylabel('Number of price checks', color='tomato')
ax2.set_ylabel('Active counters', color='steelblue')
ax1.tick_params(axis='y', colors='tomato')
ax2.tick_params(axis='y', colors='steelblue')
plt.title('Price checks vs active counters per hour')
plt.tight_layout()
plt.show()


## Shift duration for cashiers

In [ ]:
per_cashier = shifts.groupby('case:concept:name')['duration_hours'].agg(['min', 'max', 'mean']).round(2).sort_values('mean')

fig, ax = plt.subplots(figsize=(12, 5))

x = range(len(per_cashier))
ax.vlines(x, per_cashier['min'], per_cashier['max'], color='steelblue', linewidth=2, label='min–max range')
ax.scatter(x, per_cashier['mean'], color='tomato', zorder=3, label='mean')
ax.scatter(x, per_cashier['min'], color='steelblue', marker='_', s=100, zorder=3)
ax.scatter(x, per_cashier['max'], color='steelblue', marker='_', s=100, zorder=3)

ax.set_xticks(list(x))
ax.set_xticklabels(per_cashier.index.astype(str), rotation=90)
ax.set_xlabel('Cashier ID')
ax.set_ylabel('Shift duration (hours)')
ax.set_title('Shift duration per cashier (min, mean, max)')
ax.legend()
plt.tight_layout()
plt.show()


Number of shits each cashier has 

In [ ]:
shifts.groupby('case:concept:name').size().sort_values()


In [ ]:
shifts['start_h'] = shifts['shift_start'].dt.hour + shifts['shift_start'].dt.minute / 60
shifts['end_h']   = shifts['shift_end'].dt.hour   + shifts['shift_end'].dt.minute   / 60

# group by interval and count
grouped = (
    shifts.assign(
        start_h_r=shifts['start_h'].round(1),
        end_h_r=shifts['end_h'].round(1)
    )
    .groupby(['start_h_r', 'end_h_r'])
    .size()
    .reset_index(name='count')
    .sort_values('start_h_r')
)

fig, ax = plt.subplots(figsize=(11, len(grouped) * 0.5 + 1))

for i, row in grouped.iterrows():
    ax.barh(i, row['end_h_r'] - row['start_h_r'], left=row['start_h_r'],
            height=0.6, color='steelblue', edgecolor='white')
    ax.text((row['start_h_r'] + row['end_h_r']) / 2, i, str(row['count']),
            ha='center', va='center', color='white', fontweight='bold', fontsize=9)

ax.set_yticks(range(len(grouped)))
ax.set_yticklabels([f"{row['start_h_r']:.0f}h – {row['end_h_r']:.1f}h"
                    for _, row in grouped.iterrows()])
ax.set_xlabel('Hour of day')
ax.set_title('Shift intervals and number of occurrences')
ax.set_xticks(range(int(grouped['start_h_r'].min()), int(grouped['end_h_r'].max()) + 1))
ax.grid(axis='x', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()


## How many customers can a cashier help each hour on average?

Time interval from the last scan item to end price check 

In [ ]:
# Time from last Scan Item to End price check, per 5-min bin
pc_ends = (counter_df[counter_df['concept:name'] == 'End price check']
           [['case:concept:name', 'time:timestamp']]
           .rename(columns={'time:timestamp': 'end_pc_time'})
           .sort_values('end_pc_time'))

scan_items = (counter_df[counter_df['concept:name'] == 'Scan Item']
              [['case:concept:name', 'time:timestamp']]
              .rename(columns={'time:timestamp': 'scan_time'})
              .sort_values('scan_time'))

# For each End price check, find the last Scan Item before it in the same counter trace
result = pd.merge_asof(
    pc_ends, scan_items,
    left_on='end_pc_time', right_on='scan_time',
    by='case:concept:name',
    direction='backward'
)

result['duration_sec'] = (result['end_pc_time'] - result['scan_time']).dt.total_seconds().fillna(0)

result['bin_min'] = (
    result['end_pc_time'].dt.hour * 60 +
    (result['end_pc_time'].dt.minute // 5) * 5
)

by_bin = result.groupby('bin_min')['duration_sec'].agg(['mean', 'std', 'count'])

bins   = sorted(result['bin_min'].unique())
labels = [f'{b//60}:{b%60:02d}' for b in bins]
x      = range(len(bins))

fig, axes = plt.subplots(2, 1, figsize=(16, 7), sharex=True,
                         gridspec_kw={'height_ratios': [3, 1]})

axes[0].bar(x, by_bin['mean'], width=0.6, color='steelblue', alpha=0.7, edgecolor='white')
axes[0].errorbar(x, by_bin['mean'],
                 yerr=[np.minimum(by_bin['std'], by_bin['mean']), by_bin['std']],
                 fmt='none', color='black', capsize=4, linewidth=1)
for i, mean in enumerate(by_bin['mean']):
    axes[0].text(i, mean + 0.5, f'{mean:.1f}s', ha='center', va='bottom', fontsize=7)
axes[0].set_ylabel('Avg duration (seconds)')
axes[0].set_title('Time from last Scan Item to End price check per 5-min bin\n(0 if no Scan Item preceded the price check)')
axes[0].grid(True, alpha=0.3, axis='y')

axes[1].bar(x, by_bin['count'], width=0.6, color='seagreen', alpha=0.7, edgecolor='white')
for i, count in enumerate(by_bin['count']):
    axes[1].text(i, count + 0.3, str(int(count)), ha='center', va='bottom', fontsize=7)
axes[1].set_ylabel('Count (price checks)')
axes[1].set_xlabel('Time of day')
axes[1].set_xticks(list(x))
axes[1].set_xticklabels(labels, rotation=45, fontsize=7)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()


each day individually now 

In [ ]:
result['date'] = result['end_pc_time'].dt.date
result['bin_min'] = (
    result['end_pc_time'].dt.hour * 60 +
    (result['end_pc_time'].dt.minute // 15) * 15
)

all_dates = sorted(result['date'].unique())
num_days  = len(all_dates)
NCOLS = 7
nrows = (num_days + NCOLS - 1) // NCOLS

all_by_bin = {}
for day in all_dates:
    day_data = result[result['date'] == day]
    by_bin   = day_data.groupby('bin_min')['duration_sec'].agg(['mean', 'count'])
    all_by_bin[day] = by_bin

global_max = max(
    by_bin['mean'].max() for by_bin in all_by_bin.values() if len(by_bin) > 0
) * 1.2

fig, axes = plt.subplots(nrows, NCOLS, figsize=(21, 3.5 * nrows), squeeze=False)
axes_flat = axes.flatten()

for idx, day in enumerate(all_dates):
    ax     = axes_flat[idx]
    by_bin = all_by_bin[day]

    if len(by_bin) == 0:
        ax.set_title(str(day), fontsize=7, fontweight='bold', pad=2)
        continue

    bins   = sorted(by_bin.index)
    labels = [f'{b//60}:{b%60:02d}' for b in bins]
    x      = range(len(bins))

    ax.bar(x, by_bin['mean'], width=0.6, color='steelblue', alpha=0.7, edgecolor='white')
    ax.set_ylim(0, global_max)
    ax.set_xticks(list(x))
    ax.set_xticklabels(labels, rotation=45, fontsize=5)
    ax.tick_params(axis='y', labelsize=5)
    ax.set_title(str(day), fontsize=7, fontweight='bold', pad=2)
    ax.grid(True, alpha=0.3, axis='y')

for idx in range(num_days, len(axes_flat)):
    axes_flat[idx].set_visible(False)

plt.suptitle('Time from last Scan Item to End price check per 15-min bin, per day',
             fontsize=12, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.99])
plt.show()


Can price checks start one after the other without the first one finishing?

In [ ]:
pc_events = (clerk_df[clerk_df['concept:name'].isin(['Start Price Check', 'End price check'])]
             .sort_values(['case:concept:name', 'time:timestamp']))

violations = []

for clerk, group in pc_events.groupby('case:concept:name'):
    in_check = False
    for _, row in group.iterrows():
        if row['concept:name'] == 'Start Price Check':
            if in_check:
                violations.append({
                    'clerk':     clerk,
                    'timestamp': row['time:timestamp'],
                    'issue':     'Start Price Check while previous one never ended'
                })
            in_check = True
        elif row['concept:name'] == 'End price check':
            in_check = False

print(f"Violations found: {len(violations)}")
if len(violations) > 0:
    print(pd.DataFrame(violations))
else:
    print("No violations — every Start Price Check is properly closed by an End price check before the next one.")


Clerk activity between 14 and 15 

In [ ]:
tz = clerk_df['time:timestamp'].dt.tz
clerks = sorted(clerk_df['case:concept:name'].unique())

# Match price check start/end pairs
pc_start = (clerk_df[clerk_df['concept:name'] == 'Start Price Check']
            [['case:concept:name', 'time:timestamp']]
            .rename(columns={'time:timestamp': 'start_time'})
            .sort_values('start_time'))
pc_end   = (clerk_df[clerk_df['concept:name'] == 'End price check']
            [['case:concept:name', 'time:timestamp']]
            .rename(columns={'time:timestamp': 'end_time'})
            .sort_values('end_time'))

pc_start['rank'] = pc_start.groupby('case:concept:name').cumcount()
pc_end['rank']   = pc_end.groupby('case:concept:name').cumcount()
clerk_pc_pairs   = pc_start.merge(pc_end, on=['case:concept:name', 'rank'])

cleanups = clerk_df[clerk_df['concept:name'] == 'Cleanup abandoned item'][
    ['case:concept:name', 'time:timestamp']
].copy()

all_dates = sorted(clerk_df['time:timestamp'].dt.date.unique())
num_days  = len(all_dates)
NCOLS = 7
nrows = (num_days + NCOLS - 1) // NCOLS

fig, axes = plt.subplots(nrows, NCOLS, figsize=(21, 4 * nrows), squeeze=False)
axes_flat = axes.flatten()

bar_handle     = None
scatter_handle = None

for idx, day in enumerate(all_dates):
    ax        = axes_flat[idx]
    day_start = pd.Timestamp(str(day), tz=tz) + pd.Timedelta(hours=14)
    day_end   = pd.Timestamp(str(day), tz=tz) + pd.Timedelta(hours=15)

    day_pc = clerk_pc_pairs[
        (clerk_pc_pairs['start_time'] >= day_start) &
        (clerk_pc_pairs['start_time'] <  day_end)
    ]
    day_cleanup = cleanups[
        (cleanups['time:timestamp'] >= day_start) &
        (cleanups['time:timestamp'] <  day_end)
    ]

    for y, clerk in enumerate(clerks):
        # Price check intervals
        for _, row in day_pc[day_pc['case:concept:name'] == clerk].iterrows():
            start_min = (row['start_time'] - day_start).total_seconds() / 60
            end_min   = (row['end_time']   - day_start).total_seconds() / 60
            bh = ax.barh(y, end_min - start_min, left=start_min, height=0.5,
                         color='steelblue', alpha=0.8)
            if bar_handle is None:
                bar_handle = bh[0]

        # Cleanup item events
        for _, row in day_cleanup[day_cleanup['case:concept:name'] == clerk].iterrows():
            t  = (row['time:timestamp'] - day_start).total_seconds() / 60
            sc = ax.scatter(t, y, color='tomato', s=30, marker='|', zorder=5, linewidths=1.5)
            if scatter_handle is None:
                scatter_handle = sc

    ax.set_xlim(0, 60)
    ax.set_xticks([0, 15, 30, 45, 60])
    ax.set_xticklabels(['14:00', '14:15', '14:30', '14:45', '15:00'], fontsize=5)
    ax.set_yticks(range(len(clerks)))
    ax.set_yticklabels([str(c) for c in clerks], fontsize=5)
    ax.set_title(str(day), fontsize=7, fontweight='bold', pad=2)
    ax.grid(True, alpha=0.3, axis='x')

for idx in range(num_days, len(axes_flat)):
    axes_flat[idx].set_visible(False)

if bar_handle and scatter_handle:
    fig.legend(handles=[bar_handle, scatter_handle],
               labels=['Price check (start→end)', 'Cleanup abandoned item'],
               loc='lower center', ncol=2, fontsize=10,
               bbox_to_anchor=(0.5, 0))

plt.suptitle('Clerk activity per day (14:00–15:00)', fontsize=12, fontweight='bold')
plt.tight_layout(rect=[0, 0.03, 1, 0.99])
plt.show()


clerk activity vs queue buildup


In [ ]:
tz = clerk_df['time:timestamp'].dt.tz
clerks    = sorted(clerk_df['case:concept:name'].unique())
all_dates = sorted(clerk_df['time:timestamp'].dt.date.unique())
num_days  = len(all_dates)

DAYS_PER_ROW = 4
nrows = (num_days + DAYS_PER_ROW - 1) // DAYS_PER_ROW

fig, axes = plt.subplots(nrows, DAYS_PER_ROW * 2,
                         figsize=(24, 4 * nrows), squeeze=False)

bar_handle     = None
scatter_handle = None
line_handle    = None

for idx, day in enumerate(all_dates):
    row      = idx // DAYS_PER_ROW
    col      = (idx % DAYS_PER_ROW) * 2
    ax_gantt = axes[row][col]
    ax_queue = axes[row][col + 1]

    day_start = pd.Timestamp(str(day), tz=tz) + pd.Timedelta(hours=14)
    day_end   = pd.Timestamp(str(day), tz=tz) + pd.Timedelta(hours=15)

    day_pc = clerk_pc_pairs[
        (clerk_pc_pairs['start_time'] >= day_start) &
        (clerk_pc_pairs['start_time'] <  day_end)
    ]
    day_cleanup = cleanups[
        (cleanups['time:timestamp'] >= day_start) &
        (cleanups['time:timestamp'] <  day_end)
    ]

    # gantt
    for y, clerk in enumerate(clerks):
        for _, row_ in day_pc[day_pc['case:concept:name'] == clerk].iterrows():
            s = (row_['start_time'] - day_start).total_seconds() / 60
            e = (row_['end_time']   - day_start).total_seconds() / 60
            bh = ax_gantt.barh(y, e - s, left=s, height=0.5, color='steelblue', alpha=0.8)
            if bar_handle is None:
                bar_handle = bh[0]

        for _, row_ in day_cleanup[day_cleanup['case:concept:name'] == clerk].iterrows():
            t  = (row_['time:timestamp'] - day_start).total_seconds() / 60
            sc = ax_gantt.scatter(t, y, color='tomato', s=30, marker='|', zorder=5, linewidths=1.5)
            if scatter_handle is None:
                scatter_handle = sc

    ax_gantt.set_xlim(0, 60)
    ax_gantt.set_xticks([0, 15, 30, 45, 60])
    ax_gantt.set_xticklabels(['14:00', '14:15', '14:30', '14:45', '15:00'], fontsize=5, rotation=30)
    ax_gantt.set_yticks(range(len(clerks)))
    ax_gantt.set_yticklabels([str(c) for c in clerks], fontsize=5)
    ax_gantt.set_title(str(day), fontsize=7, fontweight='bold', pad=2)
    ax_gantt.grid(True, alpha=0.3, axis='x')

    # queue length line
    eq_day = log_df[
        (log_df['concept:name'] == 'Enter Queue') &
        (log_df['time:timestamp'] >= day_start) &
        (log_df['time:timestamp'] <  day_end)
    ].sort_values('time:timestamp')

    if len(eq_day) > 0:
        t_min = [(ts - day_start).total_seconds() / 60
                 for ts in eq_day['time:timestamp']]
        lh, = ax_queue.plot(t_min, eq_day['mC'].values,
                            color='seagreen', linewidth=1.2)
        if line_handle is None:
            line_handle = lh

    ax_queue.set_xlim(0, 60)
    ax_queue.set_ylim(bottom=0)
    ax_queue.set_xticks([0, 15, 30, 45, 60])
    ax_queue.set_xticklabels(['14:00', '14:15', '14:30', '14:45', '15:00'], fontsize=5, rotation=30)
    ax_queue.set_ylabel('Customers in queue', fontsize=5)
    ax_queue.tick_params(axis='y', labelsize=5)
    ax_queue.grid(True, alpha=0.3)

# Hide unused axes
for idx in range(num_days, nrows * DAYS_PER_ROW):
    r = idx // DAYS_PER_ROW
    c = (idx % DAYS_PER_ROW) * 2
    axes[r][c].set_visible(False)
    axes[r][c + 1].set_visible(False)

handles = [h for h in [bar_handle, scatter_handle, line_handle] if h is not None]
labels  = ['Price check', 'Cleanup item', 'Queue length (customers)'][:len(handles)]
fig.legend(handles=handles, labels=labels, loc='lower center',
           ncol=3, fontsize=10, bbox_to_anchor=(0.5, 0))

plt.suptitle('Clerk activity & queue length per day (14:00–15:00)',
             fontsize=12, fontweight='bold')
plt.tight_layout(rect=[0, 0.03, 1, 0.99])
plt.show()
